# Cut Force: The Combine Trait the Stopwatch Misses
**NFL Big Data Bowl 2027, Open Track.** Jay R. Kelley, M.S. Applied Data Science (Sports Analytics), IU Indianapolis.

This notebook reproduces every number, table, and figure in the writeup from the competition data, except Figure 7 and the Appendix G context, which use public nflverse Combine data. It runs top to bottom. Every writeup figure also has an interactive version (section 16 lists where each one is), and section 20 has the Cut Force card and the highest Cut Force rep for all 107 receivers.

**Cut Force** is the peak cornering load a receiver produces at speed: lateral acceleration = speed × turn rate, reported in g.

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings('ignore')

cands = [os.environ.get('BDB_DATA', '')] + [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/players.csv', recursive=True)]
DATA = next(p for p in cands if p and os.path.exists(os.path.join(p, 'combine_tracking.csv')))
G = 9.80665 / 0.9144          # yards per second squared in one g
SEED = 2027
rng = np.random.default_rng(SEED)
print('data:', DATA)

## Contents by Big Data Bowl section

The notebook keeps its working order (data in, metric, model, checks, figures). This table maps that order onto the seven Big Data Bowl sections. Each bold heading below marks where a section starts.

| Big Data Bowl section | Where it is in this notebook |
|---|---|
| 1. Motivation | **Section 1** (next cell) |
| 2. Data description and preparation | Notebook sections 2, 4 and 5: Combine route drills, NFL game tracking, route-level targets. Cleaning rules are printed as each file loads. |
| 3. Methods and metric definition | Notebook sections 1, 6, 6b, 8 and 15: the Cut Force formula, the Bayesian hierarchical model with measurement error, the leave-one-receiver-out and leave-one-class-out checks, alternative break definitions. |
| 4. Results and interpretation | Notebook sections 3, 7, 8b, 9 and 14, and the interactive versions in 16 to 20: independence from the stopwatch, Table 1, Table 2, Figures 1 to 7, the break-phase analysis. |
| 5. Discussion and game implications | Notebook sections 10 to 13 (receivers to watch, trench check, Cut Force under load, defensive backs) and **Section 5** at the end. |
| 6. Limitations and future work | **Section 6** at the end. |
| 7. References and reproducibility | **Section 7** at the end. |


# Section 1. Motivation

Much of a receiver's value is decided at the break. The Combine measures that skill with a stopwatch, and the stopwatch only reports how long a drill took. It does not report how hard a receiver can turn while he is still running fast.

This notebook introduces **Cut Force**, a tracking-based measure of the peak cornering load a receiver produces at speed during the Combine's 10 Hz route drills. The football question is whether that number is worth putting on a scouting card. It asks three things:

1. **Is Cut Force a stable trait?** A number that changes rep to rep cannot be scouted.
2. **Does it carry over to NFL routes?** A Combine number that disappears in games is a drill artifact.
3. **Does it connect to production?** The test here is targets per route after controls, because a quarterback throws to the receiver who gets open.

Why it matters for the game: a receiver with an average 40 and a high Cut Force may play faster than his time suggests. The highest Cut Force in the data belongs to Puka Nacua, the 177th pick in 2023 and a 2025 AP first-team All-Pro, whose athleticism score ranked in the 7th percentile. If the Combine already records this trait and has never reported it, a draft room is leaving information on the table. The same formula runs on 10 Hz practice GPS, so a position coach can track it week to week.

The full written submission (the writeup attached to this notebook) carries the argument in prose. Every number, table and figure in it is reproduced below.


# Sections 2 and 3. Data description, preparation, and metric definition

Notebook sections 1 to 6b. Section 1 defines the metric on the Combine tracking. Sections 2, 4 and 5 load and clean the three data sources (Combine route drills, NFL game tracking, route-level targets) and print the filters applied. Sections 6 and 6b fit the model and check whether one receiver drives the result.


## 1. The Cut Force calculation

In [ ]:
def smooth(x, k=3):
    return pd.Series(x).rolling(k, center=True, min_periods=1).mean().to_numpy()

def cut_force(s, d, hz=10, frac=1.0, min_speed=3.0):
    """Peak lateral acceleration (yd/s^2) in one rep or route.
    frac = 1.0 takes the largest peak. frac < 1 takes the FIRST local peak that reaches frac of the maximum."""
    s = np.asarray(s, float); n = len(s)
    if n < 15:
        return None
    head = smooth(np.degrees(np.unwrap(np.radians(np.asarray(d, float)))))
    spd = smooth(s)
    lat = spd * np.radians(np.abs(np.gradient(head) * hz))          # speed x turn rate
    ok = (spd > min_speed) & (np.arange(n) > 8)                      # moving fast, past the release
    if ok.sum() < 4:
        return None
    L = np.where(ok, lat, 0.0)
    if frac >= 1.0:
        k = int(np.argmax(L))
    else:
        thr = frac * L.max()
        peaks = [i for i in range(1, n - 1) if L[i] >= thr and L[i] >= L[i - 1] and L[i] >= L[i + 1]]
        k = peaks[0] if peaks else int(np.argmax(L))
    win = lat[max(0, k - 3):k + 4]
    return dict(cut_force=float(np.mean(np.sort(win)[-3:])), entry=float(spd[max(0, k - 8):k + 1].max()), k=k)

def break_phases(s, d, hz=10, min_speed=3.0, win=10):
    """Phases around the same peak: deceleration into the break (1 s before), acceleration out (1 s after), speed kept (in g)."""
    s = np.asarray(s, float); n = len(s)
    if n < 15:
        return None
    head = smooth(np.degrees(np.unwrap(np.radians(np.asarray(d, float))))); spd = smooth(s)
    lat = spd * np.radians(np.abs(np.gradient(head) * hz)); ok = (spd > min_speed) & (np.arange(n) > 8)
    if ok.sum() < 4:
        return None
    k = int(np.argmax(np.where(ok, lat, 0.0))); acc = np.gradient(spd) * hz
    pre, post = acc[max(0, k - win):k + 1], acc[k:k + win + 1]
    entry = float(spd[max(0, k - 8):k + 1].max()); low = float(spd[max(0, k - 3):k + 4].min())
    return dict(cut=float(np.mean(np.sort(lat[max(0, k - 3):k + 4])[-3:])) / G, decel=max(0.0, -float(pre.min())) / G,
                accel=max(0.0, float(post.max())) / G if len(post) > 2 else np.nan, kept=low / entry if entry > 0 else np.nan, entry=entry)

def eb_shrink(df, col, by='nfl_id'):
    """Empirical Bayes shrinkage of player means toward the population mean."""
    g = df.groupby(by)[col].agg(['mean', 'count', 'var'])
    wv = np.nanmean(g['var']); t2 = max(1e-9, g['mean'].var() - wv / g['count'].mean())
    return (t2 / (t2 + wv / g['count'])) * g['mean']

## 2. Combine route drills

In [ ]:
players = pd.read_csv(f'{DATA}/players.csv')
WR = set(players.loc[players.nfl_position == 'WR', 'nfl_id'])
DRILLS = ['COMEBACK_ROUTE_RIGHT', 'CURL_ROUTE_RIGHT', 'DAGGER_ROUTE_LEFT', 'POST_CORNER_ROUTE_LEFT', 'POST_CORNER_ROUTE_RIGHT',
          'SLANT_ROUTE_LEFT', 'SLOT_SAIL_ROUTE_RIGHT', 'SLOT_STRIKE_ROUTE_LEFT', 'SPEED_OUT_ROUTE_LEFT', 'SPEED_OUT_LEFT']
ct = pd.read_csv(f'{DATA}/combine_tracking.csv',
                 usecols=['nfl_id', 'event_id', 'entity_type', 'time', 'drill_name', 'attempt', 'x', 'y', 's', 'dir'])
ct = ct[(ct.entity_type == 'PLAYER') & ct.drill_name.isin(DRILLS)].copy()   # BALL rows share the receiver's id
ct['drill_name'] = ct.drill_name.replace({'SPEED_OUT_LEFT': 'SPEED_OUT_ROUTE_LEFT'})
REPS = {k: d.sort_values('time') for k, d in ct.groupby(['nfl_id', 'event_id', 'attempt', 'drill_name'])}

def combine_reps(frac=1.0):
    out = []
    for (pid, ev, att, dr), d in REPS.items():
        f = cut_force(d.s.values, d.dir.values, frac=frac)
        if f:
            out.append(dict(nfl_id=pid, event_id=ev, attempt=att, drill=dr, g=f['cut_force'] / G))
    o = pd.DataFrame(out)
    # center and scale within drill AND draft class: the 2025 export differs (unquantized, shorter slant reps)
    o = o.merge(players[['nfl_id', 'draft_year']], on='nfl_id')
    grp = o.groupby(['drill', 'draft_year']).g
    o['res'] = (o.g - grp.transform('mean')) / grp.transform('std')
    return o

def split_half(o, n=500, seed=SEED):
    r_ = np.random.default_rng(seed); drills = sorted(o.drill.unique()); rs = []
    for _ in range(n):
        h = set(r_.choice(drills, len(drills) // 2, replace=False))
        a = o[o.drill.isin(h)].groupby('nfl_id').res.mean(); b = o[~o.drill.isin(h)].groupby('nfl_id').res.mean()
        j = pd.concat([a, b], axis=1).dropna(); r = j.iloc[:, 0].corr(j.iloc[:, 1]); rs.append(2 * r / (1 + r))
    return np.array(rs)

reps = combine_reps()
rs = split_half(reps); REL_C = float(np.median(rs))
print(f'{len(reps)} reps, {reps.nfl_id.nunique()} receivers, {reps.groupby("nfl_id").size().mean():.1f} reps each')
print(f'Combine reliability (Spearman-Brown): {REL_C:.2f}  IQR {np.percentile(rs, 25):.2f} to {np.percentile(rs, 75):.2f}')
print('player mean g, 10th/50th/90th pct:', reps.groupby('nfl_id').g.mean().quantile([.1, .5, .9]).round(2).tolist())
display(reps.groupby('drill').g.agg(['mean', 'std', 'size']).round(2))

## 3. Independence from the stopwatch

In [ ]:
cr = pd.read_csv(f'{DATA}/combine_results.csv').drop_duplicates('nfl_id')
P = players[players.nfl_id.isin(reps.nfl_id)][['nfl_id', 'display_name', 'draft_year', 'draft_overall_pick']].copy()
P = P.merge(eb_shrink(reps, 'res').rename('combine_cf'), left_on='nfl_id', right_index=True)
P['cf_z'] = (P.combine_cf - P.combine_cf.mean()) / P.combine_cf.std()
P = P.merge(cr[['nfl_id', 'forty', 'ten_yd_split', 'three_cone', 'short_shuttle', 'combine_weight', 'ngs_athleticism_score']], on='nfl_id', how='left')
P['pick'] = P.draft_overall_pick.fillna(262); P['lpick'] = np.log(P.pick)
rows = []
for v, lab in [('forty', '40-yard dash'), ('ten_yd_split', '10-yard split'), ('three_cone', '3-cone'), ('short_shuttle', 'Short shuttle'),
               ('combine_weight', 'Weight'), ('ngs_athleticism_score', 'NGS athleticism'), ('lpick', 'Draft slot (log pick)')]:
    q = P.dropna(subset=[v]); r, p = stats.pearsonr(q[v], q.cf_z); z = np.arctanh(r); se = 1 / np.sqrt(len(q) - 3)
    rows.append((lab, r, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se), p, len(q)))
IND = pd.DataFrame(rows, columns=['measure', 'r', 'lo', 'hi', 'p', 'n']); IND.round(3)

## 4. NFL game tracking: alignment and game Cut Force
Breaking routes are measured from the snap to the throw (or sack), or 3.5 seconds when no throw occurs.

In [ ]:
pp = pd.read_csv(f'{DATA}/player_play.csv', low_memory=False)
games = pd.read_csv(f'{DATA}/games.csv')
pp = pp[pp.game_id.isin(games.loc[games.season_type == 'REG', 'game_id'])]   # regular season only
print('regular-season player-play rows:', len(pp))
BREAK = ['OUT', 'POST', 'CORNER', 'IN', 'CROSS', 'SLANT', 'HITCH']
wr_routes = pp[pp.nfl_id.isin(WR) & pp.route_ran.notna()][['game_id', 'play_id', 'nfl_id', 'route_ran']]
brk_keys = wr_routes[wr_routes.route_ran.isin(BREAK)]
gfe, aln, gph = [], [], []
for f in sorted(glob.glob(f'{DATA}/game_tracking_*.csv')):
    parts = []
    for ch in pd.read_csv(f, usecols=['game_id', 'play_id', 'nfl_id', 'time', 'y', 's', 'dir', 'event'], chunksize=4_000_000):
        parts.append(ch[ch.nfl_id.isin(WR)])
    gt = pd.concat(parts)
    sn = gt[gt.event == 'ball_snap'].drop_duplicates(['game_id', 'play_id', 'nfl_id'])
    aln.append(sn.assign(side_dist=np.minimum(sn.y, 53.3 - sn.y))[['game_id', 'play_id', 'nfl_id', 'side_dist']])
    gt = gt.merge(brk_keys, on=['game_id', 'play_id', 'nfl_id']).sort_values(['game_id', 'play_id', 'nfl_id', 'time'])
    for (g, p, n, r), d in gt.groupby(['game_id', 'play_id', 'nfl_id', 'route_ran'], sort=False):
        ev = d.event.to_numpy(); snap = np.where(ev == 'ball_snap')[0]
        if len(snap) == 0:
            continue
        st = snap[0]; pf = np.where(np.isin(ev, ['pass_forward', 'qb_sack', 'qb_strip_sack']))[0]; pf = pf[pf > st]
        en = pf[0] + 1 if len(pf) else st + 35
        b = cut_force(d.s.to_numpy()[st:en], d.dir.to_numpy()[st:en])
        if b:
            gfe.append(dict(game_id=g, play_id=p, nfl_id=n, route=r, g=b['cut_force'] / G, entry=b['entry']))
        ph = break_phases(d.s.to_numpy()[st:en], d.dir.to_numpy()[st:en])
        if ph:
            gph.append(dict(nfl_id=n, route=r, **ph))
    print(os.path.basename(f), len(gfe))
GF = pd.DataFrame(gfe); ALIGN = pd.concat(aln)
GF['res'] = smf.ols('g ~ C(route) + entry', GF).fit().resid      # adjust for route type and entry speed

# game reliability: odd versus even routes within player
GF['half'] = GF.groupby('nfl_id').cumcount() % 2
h = GF.groupby(['nfl_id', 'half']).res.mean().unstack(); r = h[0].corr(h[1]); REL_G = 2 * r / (1 + r)
P = P.merge(eb_shrink(GF, 'res').rename('game_cf'), left_on='nfl_id', right_index=True, how='left')
q = P.dropna(subset=['game_cf']); r_t, p_t = stats.pearsonr(q.cf_z, q.game_cf)
print(f'{len(GF)} breaking routes. Game reliability {REL_G:.2f}')
print(f'Transfer r = {r_t:.2f} (p = {p_t:.3f}, n = {len(q)}). Corrected r = {r_t / np.sqrt(REL_C * REL_G):.2f}')

## 5. Route-level target data

In [ ]:
S = pp[pp.nfl_id.isin(WR) & pp.route_ran.notna() & (pp.play_nullified_by_penalty != 'Y')].merge(ALIGN, on=['game_id', 'play_id', 'nfl_id'], how='left')
S['slot'] = (S.side_dist > 13).astype(float)            # snap distance to nearest sideline, bimodal split at 13 yards
S['cov'] = S.team_coverage_man_zone.fillna('UNK'); S['ytg'] = S.yards_to_go.clip(upper=20)
S['tgt'] = S.target.astype(str).str.lower().isin(['true', '1', '1.0']).astype(int)
S = S.merge(games[['game_id', 'season']], on='game_id')   # NFL season from games.csv (a game_id date puts January games in the next year)
S = S.merge(P[['nfl_id', 'cf_z', 'lpick', 'combine_weight']], on='nfl_id').dropna(subset=['cf_z', 'slot'])
S['wt_z'] = (S.combine_weight - S.combine_weight.mean()) / S.combine_weight.std()
pl_rr = S.groupby('nfl_id').agg(rr=('tgt', 'size'), tprr=('tgt', 'mean'))
P = P.merge(pl_rr, left_on='nfl_id', right_index=True, how='left')
print(f'{len(S)} routes, {S.nfl_id.nunique()} receivers, target rate {S.tgt.mean():.3f}')
F = 'tgt ~ cf_z + C(route_ran) + slot + C(cov) + C(down) + ytg + lpick + wt_z + C(season)'
glm = smf.glm(F, S, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': S.nfl_id})
b, se = glm.params.cf_z, glm.bse.cf_z
print(f'Cluster-robust logit: OR per SD = {np.exp(b):.3f} [{np.exp(b - 1.96 * se):.3f}, {np.exp(b + 1.96 * se):.3f}], p = {glm.pvalues.cf_z:.3f}')
S['brk'] = S.route_ran.isin(BREAK).astype(int)
gi = smf.glm(F.replace('cf_z', 'cf_z*brk'), S, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': S.nfl_id})
print(f'Breaking-route interaction p = {gi.pvalues["cf_z:brk"]:.2f}')
MZ = S[S['cov'].isin(['MAN_COVERAGE', 'ZONE_COVERAGE'])].copy(); MZ['man'] = (MZ['cov'] == 'MAN_COVERAGE').astype(int)
gm = smf.glm(F.replace('cf_z', 'cf_z*man').replace(' + C(cov)', ''), MZ, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': MZ.nfl_id})
for c in ['MAN_COVERAGE', 'ZONE_COVERAGE']:
    d = MZ[MZ['cov'] == c]; mc = smf.glm(F.replace(' + C(cov)', ''), d, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': d.nfl_id})
    print(f'{c}: OR per SD = {np.exp(mc.params.cf_z):.3f}, p = {mc.pvalues.cf_z:.3f}, n = {len(d)}')
print(f'Man vs zone interaction p = {gm.pvalues["cf_z:man"]:.2f}')

## 6. Bayesian hierarchical model with measurement error (PyMC)
Observed Combine Cut Force = latent trait + drill noise, with the noise share fixed by the reliability above.
Each receiver has a random intercept. Routes are aggregated to binomial cells for speed.

In [ ]:
import pymc as pm, arviz as az
S['dn'] = S.down.clip(upper=3).astype(int)
def fit_target_model(D):
    cells = D.groupby(['nfl_id', 'route_ran', 'slot', 'cov', 'dn']).agg(n=('tgt', 'size'), y=('tgt', 'sum')).reset_index()
    pid, plist = pd.factorize(cells.nfl_id); rid, rl = pd.factorize(cells.route_ran); cid, cl = pd.factorize(cells['cov']); did, dl = pd.factorize(cells.dn)
    PL = D.groupby('nfl_id')[['cf_z', 'lpick', 'wt_z']].first().reindex(plist)
    cf_obs = ((PL.cf_z - PL.cf_z.mean()) / PL.cf_z.std()).values; lp = ((PL.lpick - PL.lpick.mean()) / PL.lpick.std()).values
    with pm.Model():
        true_cf = pm.Normal('true_cf', 0, np.sqrt(REL_C), shape=len(plist))
        pm.Normal('cf_obs', true_cf, np.sqrt(1 - REL_C), observed=cf_obs)
        a = pm.Normal('a', -1.5, 1); b_cf = pm.Normal('b_cf', 0, 0.5); b_lp = pm.Normal('b_pick', 0, 0.5); b_wt = pm.Normal('b_wt', 0, 0.5)
        sig = pm.HalfNormal('sigma_player', 0.5); z = pm.Normal('z', 0, 1, shape=len(plist))
        u = b_cf * true_cf / np.sqrt(REL_C) + b_lp * lp + b_wt * PL.wt_z.values + sig * z   # b_cf is per SD of the TRUE trait
        rte = pm.ZeroSumNormal('route', 1, shape=len(rl)); cov = pm.ZeroSumNormal('cov', 0.5, shape=len(cl))
        dwn = pm.ZeroSumNormal('down', 0.5, shape=len(dl)); b_sl = pm.Normal('b_slot', 0, 0.5)
        eta = a + u[pid] + rte[rid] + cov[cid] + dwn[did] + b_sl * cells.slot.values
        pm.Binomial('y', n=cells.n.values, p=pm.math.invlogit(eta), observed=cells.y.values)
        return pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=SEED, progressbar=False)
def report(tr, D, label):
    bc = tr.posterior.b_cf.values.ravel(); p0 = D.tgt.mean(); lg = np.log(p0 / (1 - p0))
    dpts = 1 / (1 + np.exp(-(lg + bc))) - p0
    print(f'{label}: P(b_cf > 0) = {(bc > 0).mean():.2f}. +1 SD = {np.median(dpts) * 100:.1f} pts TPRR '
          f'[{np.percentile(dpts, 2.5) * 100:.1f}, {np.percentile(dpts, 97.5) * 100:.1f}], about {np.median(dpts) * 500:.0f} targets per 500 routes')
    dm = 1 / (1 + np.exp(-(lg + bc * np.sqrt(REL_C)))) - p0   # per SD of the MEASURED Combine score (drill noise included)
    gap = 1 / (1 + np.exp(-(lg + bc * np.sqrt(REL_C) * 1.2816))) - 1 / (1 + np.exp(-(lg - bc * np.sqrt(REL_C) * 1.2816)))
    print(f'{label}, measured score: +1 SD = {np.median(dm) * 100:.1f} pts, about {np.median(dm) * 500:.0f} targets per 500 routes; '
          f'10th to 90th percentile gap about {np.median(gap) * 500:.0f} targets per 500 routes')
    return bc, p0, lg, dpts
trace = fit_target_model(S)
display(az.summary(trace, var_names=['b_cf', 'b_pick', 'b_wt', 'b_slot', 'sigma_player'], hdi_prob=0.95).round(3))
bc, p0, lg, dpts = report(trace, S, 'All receivers')

## 6b. Does one receiver drive the target result?
Puka Nacua has the highest Cut Force and the highest target rate, so every target model is refit without him.
The cluster-robust logit is also refit once without each receiver in turn.

In [ ]:
NACUA = P.loc[P.display_name == 'Puka Nacua', 'nfl_id'].iloc[0]
S_wo = S[S.nfl_id != NACUA].copy()
g_wo = smf.glm(F, S_wo, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': S_wo.nfl_id})
print(f'Logit without Nacua: OR per SD = {np.exp(g_wo.params.cf_z):.3f}, p = {g_wo.pvalues.cf_z:.3f}')
q_wo = P.dropna(subset=['game_cf']); q_wo = q_wo[q_wo.nfl_id != NACUA]
r_wo, pr_wo = stats.pearsonr(q_wo.cf_z, q_wo.game_cf); print(f'Transfer without Nacua: r = {r_wo:.2f} (p = {pr_wo:.3f}, n = {len(q_wo)})')
loo = [np.exp(smf.glm(F, S[S.nfl_id != i], family=sm.families.Binomial()).fit().params.cf_z) for i in S.nfl_id.unique()]
print(f'Leave-one-receiver-out OR range: {min(loo):.3f} to {max(loo):.3f} (median {np.median(loo):.3f})')
trace_wo = fit_target_model(S_wo); _ = report(trace_wo, S_wo, 'Without Nacua')

# Section 4. Results and interpretation

Notebook sections 7 to 9: the outcome table (Table 1), the separation check and break-definition sensitivity (Table 2), the held-out draft class, and the writeup figures. Section 3 above (independence from the stopwatch) and section 14 below (break phases) are results too.


## 7. Other outcomes (Table 1)

In [ ]:
pcs = pd.read_csv(f'{DATA}/player_career_successes.csv')
P = P.merge(pcs[['nfl_id', 'career_offensive_snaps']], on='nfl_id', how='left')
P['snaps_py'] = P.career_offensive_snaps / (2026 - P.draft_year)
T = S[S.tgt == 1].dropna(subset=['expected_points_added'])
m_epa = smf.ols('expected_points_added ~ cf_z + C(route_ran) + slot + C(cov) + C(down) + ytg + lpick + wt_z', T).fit(cov_type='cluster', cov_kwds={'groups': T.nfl_id})
T2 = T.dropna(subset=['yards_after_catch', 'expected_yards_after_catch']).copy(); T2['yacoe'] = T2.yards_after_catch - T2.expected_yards_after_catch
m_yac = smf.ols('yacoe ~ cf_z + C(route_ran) + slot + C(cov) + lpick + wt_z', T2).fit(cov_type='cluster', cov_kwds={'groups': T2.nfl_id})
Q = P.dropna(subset=['snaps_py', 'cf_z', 'combine_weight'])
m_snp = smf.ols('np.log1p(snaps_py) ~ cf_z + lpick + combine_weight + C(draft_year)', Q).fit(cov_type='HC3')
tab = [['Targets per route run', f'{p0:.1%}', np.median(dpts), np.percentile(dpts, 2.5), np.percentile(dpts, 97.5), f'P(>0) = {(bc > 0).mean():.2f}', len(S)]]
for lab, m, base, n in [('EPA per target', m_epa, T.expected_points_added.mean(), len(T)), ('YAC over expected', m_yac, T2.yacoe.mean(), len(T2)),
                        ('Offensive snaps per season (log)', m_snp, Q.snaps_py.median(), len(Q))]:
    ci = m.conf_int().loc['cf_z']; tab.append([lab, f'{base:.3f}', m.params.cf_z, ci[0], ci[1], f'p = {m.pvalues.cf_z:.3f}', n])
TABLE1 = pd.DataFrame(tab, columns=['outcome', 'baseline', 'effect_per_sd', 'lo95', 'hi95', 'evidence', 'n']); TABLE1.round(4)

## 8. Separation check and sensitivity to the break definition (Table 2)

In [ ]:
sp = S.dropna(subset=['separation_at_pass_forward', 'cushion']).copy()
ms = smf.ols('separation_at_pass_forward ~ C(route_ran)*C(cov) + C(route_ran)*slot + cushion + C(down) + ytg', sp).fit(); sp['sres'] = ms.resid
P = P.merge(eb_shrink(sp, 'sres').rename('sep_oe'), left_on='nfl_id', right_index=True, how='left')
P = P.merge(S.groupby('nfl_id').slot.mean().rename('slot_share'), left_on='nfl_id', right_index=True, how='left')
msep = smf.ols('sep_oe ~ cf_z + combine_weight + slot_share', P.dropna(subset=['sep_oe', 'combine_weight'])).fit()
print(f'Separation over expected ~ Cut Force: b = {msep.params.cf_z:+.3f}, p = {msep.pvalues.cf_z:.3f}')
sens = []
for fr in [0.6, 0.7, 0.8, 1.0]:
    o = combine_reps(fr); x = o.groupby('nfl_id').res.mean().rename('x'); X = P.merge(x, left_on='nfl_id', right_index=True)
    a1 = X.dropna(subset=['game_cf']); a2 = X[X.rr >= 100]; a3 = X.dropna(subset=['snaps_py'])
    sens.append([fr, np.median(split_half(o, 200, 1)), stats.pearsonr(a1.x, a1.game_cf)[0], stats.pearsonr(a2.x, a2.tprr)[0], stats.spearmanr(a3.x, a3.snaps_py)[0]])
pd.DataFrame(sens, columns=['threshold', 'reliability', 'transfer_r', 'tprr_r', 'snaps_rho']).round(2)

## 8b. Held-out check: leave one draft class out
Fit the target model on two draft classes, predict every route of the third, and compare against the same model without Cut Force.

In [ ]:
V = S.merge(players[['nfl_id', 'draft_year']], on='nfl_id').merge(games[['game_id', 'season']].rename(columns={'season': 'nfl_season'}), on='game_id')
V['rookie'] = ((V.nfl_season - V.draft_year) == 0).astype(int)
base = 'tgt ~ C(route_ran) + slot + C(cov) + C(down) + ytg + lpick + wt_z + rookie'
folds, held = [], []
ll = lambda p, y: -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
for yr in sorted(V.draft_year.unique()):
    tr, te = V[V.draft_year != yr], V[V.draft_year == yr].copy()
    for nm, f in [('base', base), ('full', base + ' + cf_z')]:
        te['p_' + nm] = smf.glm(f, tr, family=sm.families.Binomial()).fit().predict(te)
    held.append(te); folds.append([yr, te.nfl_id.nunique(), len(te), ll(te.p_base, te.tgt), ll(te.p_full, te.tgt)])
FOLDS = pd.DataFrame(folds, columns=['held_out_class', 'players', 'routes', 'logloss_base', 'logloss_with_cut_force'])
H = pd.concat(held)
pl = H.groupby('nfl_id').agg(rr=('tgt', 'size'), act=('tgt', 'mean'), pb=('p_base', 'mean'), yr=('draft_year', 'first'))
pl = pl[pl.rr >= 100].join(P.set_index('nfl_id').cf_z)
r_all = stats.pearsonr(pl.cf_z, pl.act - pl.pb)
print(f'Pooled held-out log loss: base {ll(H.p_base, H.tgt):.4f}, with Cut Force {ll(H.p_full, H.tgt):.4f}')
print(f'Cut Force vs held-out residual target rate: r = {r_all[0]:.2f} (p = {r_all[1]:.3f}, n = {len(pl)})')
print(pl.groupby('yr').apply(lambda d: round(stats.pearsonr(d.cf_z, d.act - d.pb)[0], 2)).rename('r_by_class').to_string())
FOLDS.round(4)

## 9. Figures

In [ ]:
NAVY, GOLD, GRAY, LIGHT = '#13294B', '#B08400', '#6B7280', '#E9ECF1'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'figure.dpi': 120, 'savefig.dpi': 220, 'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 13.5,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'axes.linewidth': 1.0,
                     'axes.grid': True, 'grid.color': '#E5E7EB', 'grid.linewidth': 0.8, 'axes.axisbelow': True})
def panel(ax, letter):
    ax.text(-0.13, 1.06, letter, transform=ax.transAxes, fontsize=17, fontweight='bold', va='bottom', color='#1A1A1A')

# Figure 1: two clean speed-out reps (single dominant break), 90th and 10th percentile
so = reps[reps.drill == 'SPEED_OUT_ROUTE_LEFT'].sort_values('g').reset_index(drop=True)
def clean(r):
    d = REPS[(r.nfl_id, r.event_id, r.attempt, 'SPEED_OUT_ROUTE_LEFT')]
    a, b = cut_force(d.s.values, d.dir.values, frac=1.0), cut_force(d.s.values, d.dir.values, frac=0.6)
    return a is not None and b is not None and a['k'] == b['k']
so = so[[clean(r) for r in so.itertuples()]].reset_index(drop=True)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for r, col, lab in [(so.iloc[int(len(so) * .9)], NAVY, 'High Cut Force'), (so.iloc[int(len(so) * .1)], GOLD, 'Low Cut Force')]:
    d = REPS[(r.nfl_id, r.event_id, r.attempt, 'SPEED_OUT_ROUTE_LEFT')]
    x, y = d.x.values - d.x.values[0], d.y.values - d.y.values[0]
    spd = smooth(d.s.values); lat = spd * np.radians(np.abs(np.gradient(smooth(np.degrees(np.unwrap(np.radians(d.dir.values))))) * 10)) / G
    k = int(np.argmax(np.where((np.arange(len(lat)) > 8) & (spd > 3), lat, -1))); j = max(5, k - 4)
    ang = np.arctan2(y[j], x[j]); x, y = np.cos(-ang) * x - np.sin(-ang) * y, np.sin(-ang) * x + np.cos(-ang) * y
    if y[min(len(y) - 1, k + 6)] < 0: y = -y
    lab = f'{lab} ({r.g:.2f} g)'
    ax[0].plot(x, y, color=col, lw=2.6, label=lab); ax[0].scatter(x[k], y[k], s=70, color=col, zorder=5, ec='white')
    ax[1].plot(np.arange(len(lat)) / 10, lat, color=col, lw=2.4, label=lab); ax[1].scatter(k / 10, lat[k], s=50, color=col, zorder=5)
ax[0].set_xlabel('Yards upfield'); ax[0].set_ylabel('Yards across'); ax[0].set_title('Route path, same Combine speed out'); ax[0].legend(frameon=False, loc='upper left'); panel(ax[0], 'A')
ax[0].text(0.98, 0.04, 'dots mark the peak cut', transform=ax[0].transAxes, ha='right', fontsize=11, color=GRAY)
ax[1].set_xlabel('Seconds into rep'); ax[1].set_ylabel('Lateral force (g)'); ax[1].set_title('Lateral force over the rep'); panel(ax[1], 'B')
fig.tight_layout()
plt.show()

# Figure 2: independence
fig, ax = plt.subplots(figsize=(7.5, 3.8)); yy = np.arange(len(IND))[::-1]
ax.axvspan(-0.1, 0.1, color=LIGHT); ax.axvline(0, color=GRAY, lw=1)
ax.hlines(yy, IND.lo, IND.hi, color=NAVY, lw=2.2); ax.scatter(IND.r, yy, color=NAVY, s=55, zorder=3)
for y_, r_, n_ in zip(yy, IND.r, IND.n): ax.text(0.62, y_, f'r = {r_:+.2f}  (n = {n_})', va='center', fontsize=11.5, color=GRAY)
ax.set_yticks(yy); ax.set_yticklabels(IND.measure); ax.set_xlim(-0.65, 0.95); ax.set_xlabel('Correlation with Combine Cut Force (95% CI)')
ax.set_title('Cut Force is not hiding inside the stopwatch'); plt.show()

# Figure 3: transfer
q = P.dropna(subset=['game_cf']); zy = (q.game_cf - q.game_cf.mean()) / q.game_cf.std(); n_brk = GF.groupby('nfl_id').size().reindex(q.nfl_id).values
fig, ax = plt.subplots(figsize=(8, 5.6))
ax.scatter(q.cf_z, zy, s=np.clip(n_brk / 18, 12, 90), color=NAVY, alpha=0.65, ec='white', lw=0.6)
bb = np.polyfit(q.cf_z, zy, 1); xs = np.linspace(q.cf_z.min(), q.cf_z.max(), 50)
bt = np.array([np.polyval(np.polyfit(*q.assign(zy=zy.values)[['cf_z', 'zy']].sample(len(q), replace=True, random_state=i).values.T, 1), xs) for i in range(1000)])
ax.fill_between(xs, *np.percentile(bt, [2.5, 97.5], axis=0), color=GOLD, alpha=0.18, lw=0)
ax.plot(xs, np.polyval(bb, xs), color=GOLD, lw=3)
ax.text(0.02, 0.97, f'r = {r_t:.2f}  (p = {p_t:.3f}, n = {len(q)})\nreliability-corrected r ≈ {r_t / np.sqrt(REL_C * REL_G):.2f}',
        transform=ax.transAxes, va='top', fontsize=11.5, color=NAVY)
for nm in ['Puka Nacua']:
    rw = q[q.display_name == nm]
    if len(rw): ax.annotate(nm, (rw.cf_z.iloc[0], zy[rw.index].iloc[0]), xytext=(-8, 12), textcoords='offset points', ha='right', fontsize=11.5, fontweight='bold', color=NAVY, arrowprops=dict(arrowstyle='-', color=NAVY, lw=0.8))
for v in [200, 600, 1200]: ax.scatter([], [], s=np.clip(v / 18, 12, 90), color=NAVY, alpha=0.65, ec='white', label=f'{v:,} routes')
ax.legend(frameon=False, fontsize=11, loc='lower right', title='NFL breaking routes', title_fontsize=11, labelcolor=GRAY)
ax.set_xlabel('Combine Cut Force (0 = class average, 1 = one step above)'); ax.set_ylabel('NFL game Cut Force (0 = average)'); ax.set_title('Combine cuts show up on Sundays'); plt.show()

# Figure 4: targets per route run
zg = np.linspace(-2, 2, 41); draws = 1 / (1 + np.exp(-(lg + np.outer(bc, zg))))
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.fill_between(zg, *np.percentile(draws, [2.5, 97.5], axis=0) * 100, color=NAVY, alpha=0.15, label='Likely range, 95%')
ax.fill_between(zg, *np.percentile(draws, [25, 75], axis=0) * 100, color=NAVY, alpha=0.25, label='Likely range, 50%')
ax.plot(zg, np.median(draws, 0) * 100, color=NAVY, lw=2.6, label='Model estimate')
qq = P[P.rr >= 100].dropna(subset=['tprr']).copy(); qq['bin'] = pd.qcut(qq.cf_z, 5)
bq = qq.groupby('bin', observed=True).apply(lambda d: pd.Series({'z': d.cf_z.mean(), 't': (d.tprr * d.rr).sum() / d.rr.sum() * 100}))
ax.scatter(bq.z, bq.t, color=GOLD, s=70, zorder=5, ec='white', label='Actual rate, receivers in fifths')
ax.set_xlabel('Combine Cut Force (0 = class average, 1 = one step above)'); ax.set_ylabel('Targets per route run (%)'); ax.set_title('Sharper Combine cuts, more targets in the NFL')
ax.text(0.99, 0.03, f'{len(S):,} regular-season routes, {S.nfl_id.nunique()} receivers', transform=ax.transAxes, ha='right', fontsize=11, color=GRAY)
ax.legend(frameon=False, fontsize=11, loc='upper left'); plt.show()

In [ ]:
# Figure: one-page Cut Force card for a single receiver
def cut_force_card(name, save=None):
    from matplotlib.patches import FancyBboxPatch
    NAVY, GOLD, GRAY, LIGHT, INK = '#13294B', '#B08400', '#6B7280', '#E9ECF1', '#1A1A1A'
    p = P.set_index('display_name').loc[name]
    pct = lambda col, v, low_good=False: (P[col].dropna() < v).mean() * 100 if not low_good else (P[col].dropna() > v).mean() * 100
    fig = plt.figure(figsize=(11, 5.6), facecolor='white')
    fig.text(-0.13, 0.93, 'CUT FORCE CARD', color='#8F6C00', fontsize=11, fontweight='bold')
    fig.text(-0.13, 0.865, f'{name}  |  WR  |  {int(p.draft_year)} draft, pick {int(p.pick)}', color=NAVY, fontsize=17, fontweight='bold')
    fig.text(-0.13, 0.82, f'Percentiles among {len(P)} receivers with Combine route-drill tracking (2023 to 2025 classes)', color=GRAY, fontsize=11.5)

    # Panel A: stopwatch vs Cut Force percentiles
    ax = fig.add_axes([0.04, 0.12, 0.30, 0.62])
    items = [('Cut Force', pct('cf_z', p.cf_z)), ('NGS athleticism', pct('ngs_athleticism_score', p.ngs_athleticism_score)),
             ('40-yard dash', None if pd.isna(p.forty) else pct('forty', p.forty, True)),
             ('3-cone', None if pd.isna(p.three_cone) else pct('three_cone', p.three_cone, True)),
             ('Draft slot (higher = earlier)', pct('lpick', p.lpick, True))]
    for i, (lab, v) in enumerate(items[::-1]):
        ax.barh(i, 100, color=LIGHT, height=0.55)
        if v is None:
            ax.text(50, i, 'did not run', ha='center', va='center', color=GRAY, fontsize=11, style='italic')
        else:
            ax.barh(i, v, color=GOLD if lab == 'Cut Force' else NAVY, height=0.55)
            ax.text(min(v + 2, 92), i, f'{v:.0f}', va='center', fontsize=12, fontweight='bold', color=INK)
    ax.set_yticks(range(len(items))); ax.set_yticklabels([x[0] for x in items[::-1]], fontsize=12)
    ax.set_xlim(0, 100); ax.set_xticks([0, 50, 100]); ax.tick_params(axis='x', labelsize=8, colors=GRAY)
    for s in ['top', 'right', 'bottom']: ax.spines[s].set_visible(False)
    ax.set_title('What the stopwatch saw vs the break', fontsize=11, loc='left', color=NAVY)

    # Panel B: drill-by-drill Cut Force against the class
    ax = fig.add_axes([0.47, 0.12, 0.24, 0.62])
    dr = sorted(reps.drill.unique()); nice = {d: d.replace('_ROUTE', '').replace('_', ' ').title() for d in dr}
    for i, d in enumerate(dr):
        cls = reps[reps.drill == d].res
        ax.plot([cls.quantile(.1), cls.quantile(.9)], [i, i], color=LIGHT, lw=7, solid_capstyle='round')
        ax.plot(cls.median(), i, '|', color=GRAY, ms=12)
        mine = reps[(reps.drill == d) & (reps.nfl_id == p.nfl_id)].res
        if len(mine): ax.scatter(mine.max(), i, color=GOLD, s=55, zorder=5, ec=NAVY, lw=0.8)
    ax.set_yticks(range(len(dr))); ax.set_yticklabels([nice[d] for d in dr], fontsize=11)
    ax.set_xlabel('Cut Force vs class (0 = class average)', fontsize=11); ax.axvline(0, color=GRAY, lw=0.6, ls=':')
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_title('Every Combine route drill', fontsize=11, loc='left', color=NAVY)
    ax.text(0.0, -0.2, 'gold = his best rep    bar = middle 80% of class', fontsize=8, color=GRAY, transform=ax.transAxes)

    # Panel C: NFL outcomes
    ax = fig.add_axes([0.75, 0.12, 0.22, 0.62]); ax.axis('off')
    q = P[P.rr >= 100]
    tiles = [('Targets per route', f'{p.tprr * 100:.1f}%', f'median of all receivers {q.tprr.median() * 100:.1f}%'),
             ('NFL game Cut Force', f'{(P.game_cf.dropna() <= p.game_cf).mean() * 100:.0f}th pct' if pd.notna(p.game_cf) else 'n/a', 'route and speed adjusted'),
             ('Routes run (reg. season)', f'{int(p.rr):,}', '2023 to 2025')]
    for i, (lab, val, sub) in enumerate(tiles):
        y = 0.80 - i * 0.33
        ax.add_patch(FancyBboxPatch((0.0, y - 0.11), 1.0, 0.26, boxstyle='round,pad=0.01,rounding_size=0.03', fc=LIGHT, ec='none', transform=ax.transAxes))
        ax.text(0.06, y + 0.08, lab, fontsize=11, color=GRAY, transform=ax.transAxes)
        ax.text(0.06, y - 0.03, val, fontsize=17, fontweight='bold', color=NAVY, transform=ax.transAxes)
        ax.text(0.06, y - 0.085, sub, fontsize=8, color=GRAY, transform=ax.transAxes)
    ax.set_title('What happened in the NFL', fontsize=11, loc='left', color=NAVY)
    if save: fig.savefig(save, dpi=220, bbox_inches='tight', facecolor='white')
    plt.show()

cut_force_card('Puka Nacua')

# Section 5, part one. Companion checks that feed the discussion

Notebook sections 10 to 15: receivers to watch, the trench check, Cut Force under load, defensive backs, break phases, and alternative break definitions. These are the appendices of the writeup. Section 5 at the end of the notebook draws the game implications from them.


## 10. Receivers to watch (Appendix B in the writeup)
The ten highest Combine Cut Force scores, with NFL results so far, and the early picks with the lowest Cut Force.
Pick 262 marks an undrafted receiver. Target rates are shown only for receivers with at least 100 regular-season routes.

In [ ]:
med = P.loc[P.rr >= 100, 'tprr'].median()
gpct = P.game_cf.rank(pct=True) * 100
W = P.assign(game_pct=gpct, tprr_pct=(P.tprr * 100).where(P.rr >= 100))
cols = ['display_name', 'draft_year', 'pick', 'cf_z', 'rr', 'tprr_pct', 'game_pct']
print(f'Median target rate, receivers with 100+ routes: {med:.1%}')
display(W.sort_values('cf_z', ascending=False)[cols].head(10).round(1))
display(W[W.pick <= 64].sort_values('cf_z')[cols].head(5).round(1))

## 11. Trench check (Appendix C in the writeup)
Does lineman footwork in Combine drills add anything beyond the stopwatch? Drill speed is measured as peak speed per rep,
standardized within drill and draft class, then averaged across the three DL drills. Outcomes are regular-season get-off and
pressure rate per pass rush (DL with 100+ rushes) and pressure allowed per pass block (OL with 100+ blocks).

In [ ]:
DLD = ['PASS_RUSH_DRILL', 'FOUR_BAG_AGILITY_DRILL', 'FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION']
OLD = ['FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE', 'PASS_PRO-MIRROR_DRILL', 'PASS_PRO_MIRROR_DRILL']
TR = pd.concat([c[(c.entity_type == 'PLAYER') & c.drill_name.isin(DLD + OLD) & c.drill_type.isin(['SKILL_DRILLS_DL', 'SKILL_DRILLS_OL'])]
                for c in pd.read_csv(f'{DATA}/combine_tracking.csv', chunksize=2_000_000,
                                     usecols=['draft_year', 'event_id', 'nfl_id', 'entity_type', 'drill_type', 'drill_name', 'attempt', 's', 'a'])])
TR['drill'] = TR.drill_name.replace({'PASS_PRO_MIRROR_DRILL': 'PASS_PRO-MIRROR_DRILL'})
TRR = TR.groupby(['nfl_id', 'event_id', 'attempt', 'drill', 'draft_year']).agg(n=('s', 'size'), peak_s=('s', 'max'), peak_a=('a', 'max')).reset_index()
TRR = TRR[TRR.n >= 15]
print(f'Drill tracking: {TRR[TRR.drill.isin(DLD)].nfl_id.nunique()} defensive linemen, {TRR[TRR.drill.isin(OLD)].nfl_id.nunique()} offensive linemen')
for m in ['peak_s', 'peak_a']:
    grp = TRR.groupby(['drill', 'draft_year'])[m]; TRR[m + '_z'] = (TRR[m] - grp.transform('mean')) / grp.transform('std')
TZ = TRR.groupby(['nfl_id', 'drill'])[['peak_s_z', 'peak_a_z']].mean().unstack()
cres = pd.read_csv(f'{DATA}/combine_results.csv')[['nfl_id', 'forty', 'combine_weight']]
ppl = pd.read_csv(f'{DATA}/player_play.csv', usecols=['game_id', 'play_id', 'nfl_id', 'player_get_off', 'time_to_pressure', 'pressure_allowed', 'play_nullified_by_penalty'])
ppl = ppl[ppl.game_id.isin(games.loc[games.season_type == 'REG', 'game_id']) & (ppl.play_nullified_by_penalty != 'Y')]
dlo = ppl[ppl.player_get_off.notna()].groupby('nfl_id').agg(rush=('player_get_off', 'size'), getoff=('player_get_off', 'mean'), press=('time_to_pressure', lambda v: v.notna().mean()))
olo = ppl[ppl.pressure_allowed.notna()].assign(pa=lambda d: d.pressure_allowed.astype(str).str.lower().isin(['true', '1', '1.0']).astype(int)).groupby('nfl_id').agg(block=('pa', 'size'), press_allowed=('pa', 'mean'))
pos = players.set_index('nfl_id')[['nfl_position', 'draft_overall_pick']]
DLq = pd.DataFrame({'burst': TZ['peak_s_z'][DLD].mean(axis=1)}).join(dlo, how='inner').join(pos).reset_index().merge(cres, on='nfl_id', how='left')
DLq = DLq[DLq.nfl_position.isin(['DT', 'DE', 'NT', 'OLB']) & (DLq.rush >= 100)].copy()
DLq['edge'] = DLq.nfl_position.isin(['DE', 'OLB']).astype(int); DLq['lpick'] = np.log(DLq.draft_overall_pick.fillna(262))
DLq['bz'] = (DLq.burst - DLq.burst.mean()) / DLq.burst.std(); DLq['wz'] = (DLq.combine_weight - DLq.combine_weight.mean()) / DLq.combine_weight.std()
print(f'DL: n = {len(DLq)}. Drill speed vs 40: r = {DLq[["burst", "forty"]].corr().iloc[0, 1]:.2f}, vs weight: r = {DLq[["burst", "combine_weight"]].corr().iloc[0, 1]:.2f}')
for y in ['getoff', 'press']:
    for lab, f in [('raw', f'{y} ~ bz'), ('position + pick + weight', f'{y} ~ bz + edge + lpick + wz')]:
        mm = smf.wls(f, DLq, weights=DLq.rush).fit(cov_type='HC1'); print(f'  {y:7s} {lab:26s} b = {mm.params.bz:+.4f}, p = {mm.pvalues.bz:.3f}')
OLq = TZ['peak_a_z'][['FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE']].rename(columns=lambda c: 'wave_a').join(olo, how='inner').join(pos).reset_index()
OLq = OLq[OLq.nfl_position.isin(['G', 'T', 'C']) & (OLq.block >= 100)].dropna(subset=['wave_a'])
r_ol, p_ol = stats.pearsonr(OLq.wave_a, OLq.press_allowed)
print(f'OL: n = {len(OLq)}. Wave-drill peak acceleration vs pressure allowed: r = {r_ol:.2f}, p = {p_ol:.3f}')

## 12. Cut Force under load (Appendix D in the writeup)
Teams already collect wearable workload data (GPS and inertial units) every practice. The public data has no wearable feed,
so this section rebuilds the standard wearable metrics from the 10 Hz game tracking: distance, high speed running
(6.0 yd/s and up, about 5.5 m/s), sprint distance (7.5 yd/s and up), and acceleration and deceleration efforts
(about 3 m/s^2). Two questions. Does a receiver's Cut Force fade as his in-game running load builds? And do the usual
workload numbers already capture Cut Force?

In [ ]:
def play_load(gt):
    """Wearable-style load per player-play from 10 Hz tracking (s in yd/s, dis in yd)."""
    gt = gt.sort_values(['game_id', 'play_id', 'nfl_id', 'time']).copy()
    gt['s'] = gt.s.clip(upper=12.5)                                   # drop tracking glitches (> 25 mph)
    gt['ds'] = gt.groupby(['game_id', 'play_id', 'nfl_id']).s.diff() * 10
    up = gt.ds >= 3.3; dn = gt.ds <= -3.3                             # about 3 m/s^2 efforts
    k = ['game_id', 'play_id', 'nfl_id']
    first = gt.groupby(k).cumcount() == 0
    gt['acc'] = up & ~(up.shift(fill_value=False) & ~first)           # count the start of each effort
    gt['dec'] = dn & ~(dn.shift(fill_value=False) & ~first)
    gt['hsr'] = np.where(gt.s >= 6.0, gt.dis, 0.0)
    gt['spr'] = np.where(gt.s >= 7.5, gt.dis, 0.0)
    return gt.groupby(k).agg(dist=('dis', 'sum'), hsr=('hsr', 'sum'), spr=('spr', 'sum'),
                             acc=('acc', 'sum'), dec=('dec', 'sum')).reset_index()

rk = pp[pp.nfl_id.isin(P.nfl_id) & pp.route_ran.notna()][['game_id', 'play_id', 'nfl_id', 'possession_team', 'quarter']]
LD = []
for f in sorted(glob.glob(f'{DATA}/game_tracking_*.csv')):
    parts = [ch[ch.nfl_id.isin(WR)] for ch in pd.read_csv(f, usecols=['game_id', 'play_id', 'nfl_id', 'time', 's', 'dis'], chunksize=4_000_000)]
    LD.append(play_load(pd.concat(parts).merge(rk[['game_id', 'play_id', 'nfl_id']], on=['game_id', 'play_id', 'nfl_id'])))
LD = pd.concat(LD)
RL = rk.merge(LD, on=['game_id', 'play_id', 'nfl_id'], how='left').sort_values(['game_id', 'nfl_id', 'play_id'])
for c in ['dist', 'hsr', 'spr', 'acc', 'dec']:
    RL[c] = RL[c].fillna(RL[c].median())
RL['eff'] = RL.acc + RL.dec
gg = RL.groupby(['game_id', 'nfl_id'])
RL['cum_hsr'] = gg.hsr.cumsum() - RL.hsr                              # high speed yards already run this game
RL['k'] = gg.cumcount()                                               # routes already run this game
X12 = GF.merge(RL, on=['game_id', 'play_id', 'nfl_id']).merge(P[['nfl_id', 'cf_z']], on='nfl_id')
for c in ['cum_hsr', 'k']:
    X12[c + '_z'] = (X12[c] - X12[c].mean()) / X12[c].std()
X12['late'] = (X12.quarter >= 4).astype(int)
bsd = X12.groupby('nfl_id').res.mean().std()                          # spread of player averages, the scouting scale
print(f'{len(X12)} breaking routes, {X12.nfl_id.nunique()} receivers. Per game: {gg.size().mean():.1f} routes, {gg.hsr.sum().mean():.0f} high speed yards')
LOAD = {}
for lab, f, t in [('per SD of high speed yards already run', 'res ~ cum_hsr_z + C(nfl_id)', 'cum_hsr_z'),
                  ('per SD of routes already run', 'res ~ k_z + C(nfl_id)', 'k_z'),
                  ('4th quarter vs earlier', 'res ~ late + C(nfl_id)', 'late'),
                  ('load x Combine Cut Force', 'res ~ cum_hsr_z * cf_z + C(nfl_id)', 'cum_hsr_z:cf_z')]:
    mm = smf.ols(f, X12).fit(cov_type='cluster', cov_kwds={'groups': X12.nfl_id}); lo, hi = mm.conf_int().loc[t] / bsd * 100
    LOAD[t] = (mm.params[t] / bsd * 100, lo, hi, mm.pvalues[t])
    print(f'  {lab:40s} {LOAD[t][0]:+5.1f}% of a player SD [{lo:+.1f}, {hi:+.1f}], p = {mm.pvalues[t]:.3f}')
X12['res_w'] = X12.res - X12.groupby('nfl_id').res.transform('mean')
X12['q'] = pd.qcut(X12.cum_hsr, 5, labels=False)
QL = X12.groupby('q').agg(cum=('cum_hsr', 'median'), d=('res_w', 'mean'), se=('res_w', lambda v: v.std() / np.sqrt(len(v))))
print('High speed yards already run, quintile medians:', QL.cum.round().astype(int).tolist(), '| Cut Force vs own average, % of a player SD:', (QL.d / bsd * 100).round(1).tolist())
WL = RL.groupby('nfl_id').agg(n=('hsr', 'size'), hsr_rt=('hsr', 'mean'), spr_rt=('spr', 'mean'), eff_rt=('eff', 'mean')).reset_index()
WL = WL[WL.n >= 100].merge(P[['nfl_id', 'cf_z', 'forty']], on='nfl_id')
WCOR = {c: (stats.pearsonr(WL.cf_z, WL[c])[0], stats.pearsonr(WL.dropna(subset=['forty']).forty, WL.dropna(subset=['forty'])[c])[0]) for c in ['hsr_rt', 'spr_rt', 'eff_rt']}
for c, (rc, rf) in WCOR.items():
    print(f'  {c}: vs Combine Cut Force r = {rc:+.2f}, vs 40 time r = {rf:+.2f} (n = {len(WL)})')

# Figure 5 (same cell, so the plot always has QL, WCOR, and bsd)
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.35, 1]})
yv = QL.d / bsd * 100; ye = 1.96 * QL.se / bsd * 100
ax[0].axhspan(-10, 10, color=LIGHT); ax[0].axhline(0, color=GRAY, lw=1)
ax[0].errorbar(QL.cum, yv, yerr=ye, color=NAVY, marker='o', ms=7, lw=2.2, capsize=4)
ax[0].set_xlabel('High speed yards already run in the game'); ax[0].set_ylabel('Cut Force vs own average\n(% of a player SD)')
ax[0].set_ylim(-40, 40); ax[0].set_title('Cut Force holds as game load builds'); panel(ax[0], 'A')
ax[0].text(0.98, 0.05, 'shaded band = within 10% of a player SD', transform=ax[0].transAxes, ha='right', fontsize=11, color=GRAY)
labs = ['High speed yd\nper route', 'Sprint yd\nper route', 'Accel + decel\nefforts per route']
xx = np.arange(3); w = 0.36
ax[1].bar(xx - w / 2, [abs(WCOR[c][1]) for c in WCOR], w, color=GOLD, label='40 time')
ax[1].bar(xx + w / 2, [abs(WCOR[c][0]) for c in WCOR], w, color=NAVY, label='Combine Cut Force')
ax[1].set_xticks(xx); ax[1].set_xticklabels(labs, fontsize=11); ax[1].set_ylim(0, 0.6); ax[1].set_ylabel('|r| with workload metric')
ax[1].set_title('Workload numbers track speed, not cuts'); panel(ax[1], 'B'); ax[1].legend(frameon=False, fontsize=11, loc='upper right')
plt.tight_layout(); plt.savefig('fig5_load.png', dpi=220, bbox_inches='tight', facecolor='white'); plt.show()

## 13. Defensive backs (Appendix E in the writeup)
The prompt's first example pairs receivers with defensive backs, so the same metric is tested on the 118 DBs.
Combine side: Cut Force in the six DB movement drills, standardized within drill and draft class.
Game side: DB Cut Force on every regular-season coverage snap, and separation at the throw in man coverage when the
nearest rookie receiver at the snap (within 12 yards) is also in the tracking data.

In [ ]:
DBS = set(players.loc[players.nfl_position.isin(['CB', 'SS', 'FS', 'DB']), 'nfl_id'])
DBD = ['BACK_PEDAL_AND_90_DEGREE_BREAK_TO_BALL', 'BACK_PEDAL_AND_TRANSITION_45_DEGREE_REACTION', 'BOX_DRILL', 'W_DRILL',
       'TERYL_AUSTIN_DRILL_1', 'TERYL_AUSTIN_DRILL_2']
cdb = pd.concat([c[(c.entity_type == 'PLAYER') & c.nfl_id.isin(DBS) & c.drill_name.isin(DBD)]
                 for c in pd.read_csv(f'{DATA}/combine_tracking.csv', chunksize=2_000_000,
                                      usecols=['nfl_id', 'event_id', 'entity_type', 'time', 'drill_name', 'attempt', 'draft_year', 's', 'dir'])])
dbr = []
for (pid, ev, att, dr, yr), d in cdb.groupby(['nfl_id', 'event_id', 'attempt', 'drill_name', 'draft_year']):
    f = cut_force(d.sort_values('time').s.values, d.sort_values('time').dir.values)
    if f:
        dbr.append(dict(nfl_id=pid, drill=dr, draft_year=yr, g=f['cut_force'] / G))
dbr = pd.DataFrame(dbr); gq = dbr.groupby(['drill', 'draft_year']).g
dbr['res'] = (dbr.g - gq.transform('mean')) / gq.transform('std')
REL_DB = float(np.median(split_half(dbr, 500)))
PD = players[players.nfl_id.isin(DBS)][['nfl_id', 'display_name', 'draft_overall_pick']].merge(eb_shrink(dbr, 'res').rename('db_cf'), left_on='nfl_id', right_index=True)
PD['db_cf_z'] = (PD.db_cf - PD.db_cf.mean()) / PD.db_cf.std()
PD = PD.merge(cres, on='nfl_id', how='left')
print(f'DB Combine Cut Force: {len(dbr)} reps, {dbr.nfl_id.nunique()} DBs, reliability {REL_DB:.2f} (receivers {REL_C:.2f})')

cov = pp[pp.nfl_id.isin(DBS) & pp.coverage_assignment.notna()][['game_id', 'play_id', 'nfl_id', 'coverage_assignment']]
wrr = pp[pp.nfl_id.isin(WR) & pp.route_ran.notna()][['game_id', 'play_id', 'nfl_id', 'route_ran']]
covk = cov.set_index(['game_id', 'play_id', 'nfl_id']).coverage_assignment.to_dict()
wrk = wrr.set_index(['game_id', 'play_id', 'nfl_id']).route_ran.to_dict()
DF, DM = [], []
for f in sorted(glob.glob(f'{DATA}/game_tracking_*.csv')):
    parts = [ch[ch.nfl_id.isin(DBS | WR)] for ch in pd.read_csv(f, usecols=['game_id', 'play_id', 'nfl_id', 'time', 'x', 'y', 's', 'dir', 'event'], chunksize=4_000_000)]
    gt = pd.concat(parts).merge(cov[['game_id', 'play_id']].drop_duplicates(), on=['game_id', 'play_id']).sort_values(['game_id', 'play_id', 'nfl_id', 'time'])
    for (gm, pl_), play in gt.groupby(['game_id', 'play_id'], sort=False):
        trk = {n: d for n, d in play.groupby('nfl_id', sort=False)}
        wrs = {n: d for n, d in trk.items() if (gm, pl_, n) in wrk}
        for n, d in trk.items():
            ca = covk.get((gm, pl_, n))
            if ca is None:
                continue
            e = d.event.to_numpy(); sn = np.where(e == 'ball_snap')[0]
            if len(sn) == 0:
                continue
            st = sn[0]; pf = np.where(np.isin(e, ['pass_forward', 'qb_sack', 'qb_strip_sack']))[0]; pf = pf[pf > st]
            en = pf[0] + 1 if len(pf) else st + 35
            r = cut_force(d.s.to_numpy()[st:en], d.dir.to_numpy()[st:en])
            if r:
                DF.append(dict(nfl_id=n, cov=ca, g=r['cut_force'] / G, entry=r['entry'], has_pf=int(len(pf) > 0)))
            if ca == 'MAN' and wrs:
                xs, ys = d.x.to_numpy()[st], d.y.to_numpy()[st]; best = None
                for w, wd in wrs.items():
                    ws = np.where(wd.event.to_numpy() == 'ball_snap')[0]
                    if len(ws) == 0:
                        continue
                    dd = np.hypot(wd.x.to_numpy()[ws[0]] - xs, wd.y.to_numpy()[ws[0]] - ys)
                    if best is None or dd < best[1]:
                        best = (w, dd, wd)
                if best and best[1] <= 12:
                    w, dsn, wd = best; wpf = np.where(wd.event.to_numpy() == 'pass_forward')[0]
                    if len(wpf) and len(pf):
                        i = min(pf[0], len(d) - 1); j = min(wpf[0], len(wd) - 1)
                        DM.append(dict(db=n, wr=w, snap_dist=dsn, route=wrk[(gm, pl_, w)],
                                       sep=float(np.hypot(wd.x.to_numpy()[j] - d.x.to_numpy()[i], wd.y.to_numpy()[j] - d.y.to_numpy()[i]))))
DF = pd.DataFrame(DF); DM = pd.DataFrame(DM)
DF['covg'] = DF['cov'].str.replace(r'_(LEFT|RIGHT)$', '', regex=True)
DF['res'] = smf.ols('g ~ C(covg) + entry + has_pf', DF).fit().resid
DF['half'] = DF.groupby('nfl_id').cumcount() % 2
h = DF.groupby(['nfl_id', 'half']).res.mean().unstack(); r = h[0].corr(h[1]); REL_DBG = 2 * r / (1 + r)
PD = PD.merge(eb_shrink(DF, 'res').rename('db_game_cf'), left_on='nfl_id', right_index=True, how='left').merge(DF.groupby('nfl_id').size().rename('snaps'), left_on='nfl_id', right_index=True, how='left')
qd = PD[PD.snaps >= 100].dropna(subset=['db_game_cf']); r_db, p_db = stats.pearsonr(qd.db_cf_z, qd.db_game_cf)
print(f'{len(DF)} coverage snaps, {DF.nfl_id.nunique()} DBs. Game reliability {REL_DBG:.2f} (receivers {REL_G:.2f})')
print(f'DB transfer r = {r_db:.2f} (p = {p_db:.3f}, n = {len(qd)})')
DM = DM.merge(PD[['nfl_id', 'db_cf_z', 'forty']], left_on='db', right_on='nfl_id', how='left')
T10 = DM[DM.sep < 10].copy(); T10['f10'] = T10.forty * 10                  # drop loose pairings
print(f'Man coverage matchups: {len(DM)} ({DM.db.nunique()} DBs), {len(T10)} within 10 yards at the throw')
for lab, x in [('40 time, per 0.1 s', 'f10'), ('Combine DB Cut Force, per SD', 'db_cf_z')]:
    t = T10.dropna(subset=[x]); mm = smf.ols(f'sep ~ {x} + C(route) + snap_dist', t).fit(cov_type='cluster', cov_kwds={'groups': t.db})
    lo, hi = mm.conf_int().loc[x]
    print(f'  separation ~ {lab:30s} b = {mm.params[x]:+.2f} yd [{lo:+.2f}, {hi:+.2f}], p = {mm.pvalues[x]:.3f}, n = {len(t)}')

## 14. Brakes, cut, or burst? Break phases (Section 5 and Figure 3 in the writeup)
The prompt's route-running example asks about deceleration into breaks and acceleration out of cuts. Each rep and route is split
into phases around the same peak: deceleration over the second before it, Cut Force at it, acceleration over the second after it,
and the share of entry speed kept through it. Every phase gets the same checks as Cut Force.

In [ ]:
PH_REL, PH_RELG = {}, {}
PH_LAB = {'decel': 'Deceleration into the break', 'cut': 'Cut Force (at the break)', 'accel': 'Acceleration out of the break', 'kept': 'Speed kept through the break'}
cph = []
for (pid, ev, att, dr), d in REPS.items():
    f = break_phases(d.s.values, d.dir.values)
    if f:
        cph.append(dict(nfl_id=pid, drill=dr, **f))
CPH = pd.DataFrame(cph).merge(players[['nfl_id', 'draft_year']], on='nfl_id'); GPH = pd.DataFrame(gph)
PP = P[['nfl_id', 'forty', 'ten_yd_split', 'three_cone', 'short_shuttle', 'combine_weight', 'lpick', 'sep_oe', 'slot_share']].copy()
for m in PH_LAB:
    gq = CPH.groupby(['drill', 'draft_year'])[m]; CPH['res'] = (CPH[m] - gq.transform('mean')) / gq.transform('std')
    PP = PP.merge(eb_shrink(CPH, 'res').rename(m + '_cz'), left_on='nfl_id', right_index=True, how='left')
    PP[m + '_cz'] = (PP[m + '_cz'] - PP[m + '_cz'].mean()) / PP[m + '_cz'].std()
    PH_REL[m] = float(np.median(split_half(CPH)))
    GPH['gres'] = smf.ols(f'{m} ~ C(route) + entry', GPH.dropna(subset=[m])).fit().resid
    GPH['half'] = GPH.groupby('nfl_id').cumcount() % 2
    hh = GPH.groupby(['nfl_id', 'half']).gres.mean().unstack(); rr_ = hh[0].corr(hh[1])
    PH_RELG[m] = 2 * rr_ / (1 + rr_)
    PP = PP.merge(eb_shrink(GPH.dropna(subset=['gres']), 'gres').rename(m + '_gz'), left_on='nfl_id', right_index=True, how='left')
rows = []
for m, lab in PH_LAB.items():
    q = PP.dropna(subset=[m + '_cz', m + '_gz']); rt = stats.pearsonr(q[m + '_cz'], q[m + '_gz'])
    sw = max(abs(PP[[v, m + '_cz']].dropna().corr().iloc[0, 1]) for v in ['forty', 'ten_yd_split', 'three_cone', 'short_shuttle'])
    qs = PP.dropna(subset=['sep_oe', 'combine_weight', m + '_cz']); ms_ = smf.ols(f'sep_oe ~ {m}_cz + combine_weight + slot_share', qs).fit()
    qg = PP.dropna(subset=['sep_oe', 'combine_weight', m + '_gz']).copy(); qg['gz'] = (qg[m + '_gz'] - qg[m + '_gz'].mean()) / qg[m + '_gz'].std()
    mg_ = smf.ols('sep_oe ~ gz + combine_weight + slot_share', qg).fit()
    rows.append([lab, PH_REL[m], PH_RELG[m], rt[0], rt[1], sw, ms_.params[m + '_cz'], ms_.pvalues[m + '_cz'], mg_.pvalues['gz'], len(q)])
PHASES = pd.DataFrame(rows, columns=['phase', 'combine_rel', 'game_rel', 'transfer_r', 'transfer_p', 'stopwatch_max_abs_r', 'sep_b', 'sep_p', 'game_sep_p', 'n'])
print('3-cone vs deceleration into the break: r =', round(PP[['three_cone', 'decel_cz']].dropna().corr().iloc[0, 1], 2))
SP = S.merge(PP[['nfl_id'] + [m + '_cz' for m in PH_LAB]].rename(columns={m + '_cz': m + '_ph' for m in PH_LAB}), on='nfl_id')
gj = smf.glm(F.replace('cf_z', ' + '.join(m + '_ph' for m in PH_LAB)), SP, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': SP.nfl_id})
print('Joint target model, odds ratio per SD:', {m: (round(np.exp(gj.params[m + '_ph']), 3), round(gj.pvalues[m + '_ph'], 3)) for m in PH_LAB})
PHASES.round(3)

In [ ]:
# Figure 3: break-phase scorecard (the bottom row is defensive back Cut Force from section 13)
from matplotlib.colors import LinearSegmentedColormap
tdb = T10.dropna(subset=['db_cf_z']); mdb = smf.ols('sep ~ db_cf_z + C(route) + snap_dist', tdb).fit(cov_type='cluster', cov_kwds={'groups': tdb.db})
labs = list(PHASES.phase) + ['Defensive back Cut Force']
vals = [[r.combine_rel, r.game_rel, r.transfer_r, 1 - r.stopwatch_max_abs_r] for r in PHASES.itertuples()] + [[REL_DB, REL_DBG, r_db, None]]
txt = [[f'{r.combine_rel:.2f}', f'{r.game_rel:.2f}', f'{r.transfer_r:+.2f}', f'{r.stopwatch_max_abs_r:.2f}', f'{r.sep_b:+.3f} ({r.sep_p:.2f})'] for r in PHASES.itertuples()]
txt += [[f'{REL_DB:.2f}', f'{REL_DBG:.2f}', f'{r_db:+.2f}', 'not tested', f"{mdb.params.db_cf_z:+.2f} ({mdb.pvalues.db_cf_z:.2f})"]]
heads = ['Combine\nreliability', 'NFL game\nreliability', 'Carries to\nNFL routes (r)', 'Overlap with\nstopwatch drills\n(lower = more new info)', 'Separation at\nthrow, yd per SD\n(p)']
cm = LinearSegmentedColormap.from_list('c', ['#F3F4F6', '#C9D3E3', NAVY]); scl = [(0, 0.7), (0, 0.8), (-0.1, 0.35), (0.4, 1.0)]
fig, ax = plt.subplots(figsize=(12.5, 5.3)); ax.set_xlim(-0.5, 4.5); ax.set_ylim(len(labs) - 0.5, -0.5); ax.axis('off')
for i, lab in enumerate(labs):
    for j in range(5):
        v = vals[i][j] if j < 4 else None
        if v is None:
            fc, tc = 'white', (GRAY if j < 4 else '#1A1A1A')
        else:
            x = min(1, max(0, (v - scl[j][0]) / (scl[j][1] - scl[j][0]))); fc = cm(x); tc = 'white' if x > 0.6 else '#1A1A1A'
        ax.add_patch(plt.Rectangle((j - 0.48, i - 0.42), 0.96, 0.84, color=fc, ec='#E5E7EB', lw=1))
        ax.text(j, i, txt[i][j], ha='center', va='center', fontsize=14 if j < 4 else 12.5, color=tc, fontweight='bold' if i == 1 and j < 4 else 'normal')
    ax.text(-0.56, i, lab, ha='right', va='center', fontsize=13, fontweight='bold' if i == 1 else 'normal', color=NAVY if i == 1 else '#1A1A1A')
for j, c in enumerate(heads): ax.text(j, -0.75, c, ha='center', va='bottom', fontsize=11.5, color=GRAY)
ax.add_patch(plt.Rectangle((-3.55, 0.56), 8.05, 0.88, fill=False, ec=GOLD, lw=2.2, clip_on=False))
ax.text(-3.5, -1.85, 'Only the cut itself is a Combine trait that carries to NFL routes', fontsize=15.5, fontweight='bold', ha='left', va='bottom')
ax.text(-3.5, 4.75, 'Darker = better. Overlap is shaded so that less overlap with the 40, 10-yard split, 3-cone, and shuttle reads darker (more new information).\n'
        'Separation is regressed on each Combine measure with weight and slot share controlled (DB row: man-coverage matchups, route and snap distance controlled).', fontsize=10.5, color=GRAY, ha='left', va='top')
fig.subplots_adjust(left=0.27, right=0.99, top=0.80, bottom=0.12); plt.show()

## 15. Alternative break definitions and a held-out draft class (Appendix H in the writeup)

Ten definitions of the break on the same Combine reps, scored on the same checks, then a player-level leave-one-draft-class-out test of targets per route. The route-level version of the held-out test is in Section 8b.


In [ ]:
def break_measures(s, d, hz=10, min_speed=3.0):
    """Ten ways to score one rep. Peak variants, whole-turn variants, and the three phases from Section 14."""
    s = np.asarray(s, float); n = len(s)
    if n < 15:
        return None
    out = {}
    for k_s in (3, 5):
        head = smooth(np.degrees(np.unwrap(np.radians(np.asarray(d, float)))), k_s); spd = smooth(s, k_s)
        lat = spd * np.radians(np.abs(np.gradient(head) * hz)); ok = (spd > min_speed) & (np.arange(n) > 8)
        if ok.sum() < 4:
            return None
        L = np.where(ok, lat, 0.0); k = int(np.argmax(L)); win = lat[max(0, k - 3):k + 4]
        if k_s == 5:
            out['peak5'] = np.mean(np.sort(win)[-3:]) / G
            continue
        out['peak'] = np.mean(np.sort(win)[-3:]) / G
        out['single'] = lat[k] / G
        thr = 0.6 * L.max(); pk = [i for i in range(1, n - 1) if L[i] >= thr and L[i] >= L[i - 1] and L[i] >= L[i + 1]]
        kf = pk[0] if pk else k; out['first'] = np.mean(np.sort(lat[max(0, kf - 3):kf + 4])[-3:]) / G
        a = k
        while a > 0 and lat[a - 1] >= 0.5 * lat[k]:
            a -= 1
        b = k
        while b < n - 1 and lat[b + 1] >= 0.5 * lat[k]:
            b += 1
        seg = lat[a:b + 1]
        out['impulse'] = seg.sum() / hz / G; out['meanturn'] = seg.mean() / G; out['duration'] = (b - a + 1) / hz
        acc = np.gradient(spd) * hz; pre, post = acc[max(0, k - 10):k + 1], acc[k:k + 11]
        entry = spd[max(0, k - 8):k + 1].max(); low = spd[max(0, k - 3):k + 4].min()
        out['decel'] = max(0.0, -pre.min()) / G; out['accel'] = (max(0.0, post.max()) / G) if len(post) > 2 else np.nan
        out['kept'] = low / entry if entry > 0 else np.nan
    return out

ALT_LAB = {'peak': 'Peak Cut Force (primary)', 'first': 'First break only', 'single': 'Single peak frame', 'peak5': 'Five-frame smoothing',
           'meanturn': 'Mean across the turn', 'impulse': 'Integrated across the turn', 'duration': 'Turn duration',
           'decel': 'Deceleration in', 'accel': 'Acceleration out', 'kept': 'Speed kept'}
rows = []
for (pid, ev, att, dr), d in REPS.items():
    f = break_measures(d.s.values, d.dir.values)
    if f:
        rows.append(dict(nfl_id=pid, drill=dr, **f))
AR = pd.DataFrame(rows).merge(players[['nfl_id', 'draft_year']], on='nfl_id')
X = P[['nfl_id', 'display_name', 'draft_year', 'game_cf', 'tprr', 'rr', 'lpick', 'slot_share', 'combine_weight', 'forty', 'three_cone']].copy()
alt_rows, HO = [], {}
for m, lab in ALT_LAB.items():
    o = AR.dropna(subset=[m]).copy(); gq = o.groupby(['drill', 'draft_year'])[m]
    o['res'] = (o[m] - gq.transform('mean')) / gq.transform('std')
    rel = float(np.median(split_half(o, n=400)))
    Xm = X.merge(eb_shrink(o, 'res').rename('mz'), left_on='nfl_id', right_index=True); Xm['mz'] = (Xm.mz - Xm.mz.mean()) / Xm.mz.std()
    q = Xm.dropna(subset=['game_cf']); tr = stats.pearsonr(q.mz, q.game_cf)
    r40 = Xm.dropna(subset=['forty'])[['mz', 'forty']].corr().iloc[0, 1]
    q = Xm[Xm.rr >= 100].dropna(subset=['tprr']); tp = stats.pearsonr(q.mz, q.tprr)
    q = Xm[Xm.rr >= 100].dropna(subset=['tprr', 'lpick', 'slot_share', 'combine_weight']).copy(); folds, held = [], []
    for yr in sorted(q.draft_year.unique()):
        tr_, te = q[q.draft_year != yr], q[q.draft_year == yr].copy()
        te['pb'] = smf.ols('tprr ~ lpick + slot_share + combine_weight', tr_).fit().predict(te)
        te['pf'] = smf.ols('tprr ~ lpick + slot_share + combine_weight + mz', tr_).fit().predict(te)
        rr_ = stats.pearsonr(te.mz, te.tprr - te.pb)
        folds.append(dict(held_out=int(yr), n=len(te), rmse_base=np.sqrt(np.mean((te.tprr - te.pb) ** 2)) * 100,
                          rmse_with=np.sqrt(np.mean((te.tprr - te.pf) ** 2)) * 100, resid_r=rr_[0], resid_p=rr_[1]))
        held.append(te)
    H = pd.concat(held); pooled = stats.pearsonr(H.mz, H.tprr - H.pb)
    HO[m] = pd.DataFrame(folds)
    alt_rows.append([lab, rel, r40, tr[0], len(Xm.dropna(subset=['game_cf'])), tp[0], len(Xm[Xm.rr >= 100].dropna(subset=['tprr'])), pooled[0], pooled[1],
                     np.sqrt(np.mean((H.tprr - H.pb) ** 2)) * 100, np.sqrt(np.mean((H.tprr - H.pf) ** 2)) * 100])
ALT = pd.DataFrame(alt_rows, columns=['definition', 'reliability', 'r_forty', 'transfer_r', 'transfer_n', 'tprr_r', 'tprr_n', 'heldout_r', 'heldout_p', 'rmse_base', 'rmse_with'])
ALT['rmse_change'] = ALT.rmse_with - ALT.rmse_base
PK = {}
for m in ['peak', 'single', 'peak5', 'first']:
    o = AR.dropna(subset=[m]).copy(); gq = o.groupby(['drill', 'draft_year'])[m]
    o['res'] = (o[m] - gq.transform('mean')) / gq.transform('std'); PK[m] = eb_shrink(o, 'res')
print('Peak variants, player-level r with the primary:', pd.DataFrame(PK).corr().loc['peak', ['single', 'peak5', 'first']].round(2).to_dict())
print('Table H1: ten break definitions'); display(ALT.round(2))
print('Table H2: leave one draft class out, primary vs first break')
display(HO['peak'].merge(HO['first'], on=['held_out', 'n'], suffixes=('_primary', '_first')).round(3))


# Section 4, interactive. Every figure with hover, a break replay, an explorer, and route maps

Notebook sections 16 to 20. The table at the top of section 16 lists where each writeup figure's interactive version lives.


## 16. Interactive figures (Plotly)

Every figure in the writeup has an interactive version in this notebook. Hover for the receiver or the exact number, drag to zoom, and double-click to reset.

| Writeup | Interactive version |
|---|---|
| Figure 1, the break replay | section 17 (two reps, animated) and section 20 (every receiver's highest Cut Force rep) |
| Figure 2, Combine to NFL transfer | below |
| Figure 3, the break-phase scorecard | below |
| Figure 4, targets per route | below (player-level view) |
| Figure 5, the Cut Force card | section 20, for all 107 receivers |
| Figure 6, Cut Force under load | below |
| Figure 7, the historical Combine | below (needs the notebook's internet setting on, because it reads the public nflverse file) |
| Tables H1 and H2 | below |

The section 3 stopwatch check is here too. The static images above are the versions in the writeup.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as po
from IPython.display import HTML, display
# Kaggle's saved-notebook viewer loads require.js. With it on the page, plotly.js registers itself as an AMD module
# instead of the global Plotly that every figure below calls, and the charts come out blank ("Plotly is not defined").
# So define is hidden while the bundle loads once, inline, and restored right after.
display(HTML('<script type="text/javascript">window.__cfDefine = window.define; window.define = undefined;</script>'
             '<script type="text/javascript">' + po.get_plotlyjs() + '</script>'
             '<script type="text/javascript">window.define = window.__cfDefine;</script>'
             # The viewer can draw a chart before its page reaches full width, so every chart is refit once the page has loaded.
             '<script type="text/javascript">(function () { function fit() { if (!window.Plotly) return;'
             ' document.querySelectorAll(".plotly-graph-div").forEach(function (d) { try { window.Plotly.Plots.resize(d); } catch (e) {} }); }'
             ' window.addEventListener("load", function () { setTimeout(fit, 100); setTimeout(fit, 1200); });'
             ' var t; window.addEventListener("resize", function () { clearTimeout(t); t = setTimeout(fit, 150); }); })();</script>'))
def show(fig):
    display(HTML(fig.to_html(include_plotlyjs=False, full_html=False, auto_play=False, config={'displaylogo': False, 'responsive': True})))
def label(txt):
    display(HTML(f'<p style="margin:22px 0 0;font:600 15px Helvetica,Arial,sans-serif;color:#13294B">{txt}</p>'))
NV, GD = '#13294B', '#B08400'
LAY = dict(template='plotly_white', font=dict(family='Helvetica, Arial', size=13), margin=dict(l=60, r=20, t=60, b=50))

# Writeup Figure 2: Combine to game transfer, hover shows the receiver
label('Writeup Figure 2. Combine Cut Force against NFL game Cut Force')
q = P.dropna(subset=['game_cf']).copy(); q['gz'] = (q.game_cf - q.game_cf.mean()) / q.game_cf.std()
m_, b_ = np.polyfit(q.cf_z, q.gz, 1); xs = np.array([-2.5, 2.8]); r_, p_ = stats.pearsonr(q.cf_z, q.gz)
fig = go.Figure()
fig.add_trace(go.Scatter(x=q.cf_z, y=q.gz, mode='markers', name='Receivers', text=[f'{n} ({y})<br>{int(r)} breaking routes' for n, y, r in zip(q.display_name, q.draft_year, q.rr.fillna(0))],
                         marker=dict(size=6 + np.sqrt(q.rr.fillna(50)) / 3, color=NV, opacity=.75, line=dict(color='white', width=1)),
                         hovertemplate='%{text}<br>Combine z = %{x:.2f}<br>Game z = %{y:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(x=xs, y=m_ * xs + b_, mode='lines', name='Least-squares fit', line=dict(color=GD, width=2.5), hoverinfo='skip'))
fig.update_layout(title=f'Combine cuts show up on Sundays (r = {r_:.2f}, p = {p_:.3f}, n = {len(q)})', xaxis_title='Combine Cut Force (0 = class average, 1 = one step above)', yaxis_title='NFL game Cut Force (z, route and speed adjusted)', **LAY)
show(fig)

# Section 3 check (not a writeup figure): independence from the stopwatch
label('Section 3 check. Cut Force against every stopwatch measure')
fig = go.Figure(go.Bar(x=IND.r, y=IND.measure, orientation='h', marker_color=[GD if v < 0 else NV for v in IND.r], text=IND.r.round(2), textposition='outside',
                       error_x=dict(type='data', symmetric=False, array=IND.hi - IND.r, arrayminus=IND.r - IND.lo, color='#6B7280'), hovertemplate='%{y}: r = %{x:.2f}<extra></extra>'))
fig.update_layout(title='Cut Force is new information: correlation with every traditional measure (95% intervals)', xaxis=dict(range=[-0.5, 0.5], title='Pearson r with Combine Cut Force', zeroline=True), yaxis=dict(autorange='reversed'), **LAY)
show(fig)

# Writeup Figure 4, player-level view: targets per route
label('Writeup Figure 4, player-level view. Targets per route against Combine Cut Force')
q = P[P.rr >= 100].dropna(subset=['tprr']).copy(); q['q5'] = pd.qcut(q.cf_z, 5, labels=False); qm = q.groupby('q5').agg(x=('cf_z', 'mean'), y=('tprr', 'mean'))
m_, b_ = np.polyfit(q.cf_z, q.tprr, 1); r_, p_ = stats.pearsonr(q.cf_z, q.tprr)
fig = go.Figure()
fig.add_trace(go.Scatter(x=q.cf_z, y=100 * q.tprr, mode='markers', name='Receivers (100+ routes)', text=[f'{n} ({y})<br>{int(r)} routes' for n, y, r in zip(q.display_name, q.draft_year, q.rr)],
                         marker=dict(size=6 + np.sqrt(q.rr) / 4, color=NV, opacity=.7, line=dict(color='white', width=1)), hovertemplate='%{text}<br>Cut Force z = %{x:.2f}<br>targets per route = %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=qm.x, y=100 * qm.y, mode='markers', name='Quintile mean', marker=dict(symbol='square', size=13, color=GD, line=dict(color='white', width=1.5))))
fig.add_trace(go.Scatter(x=xs, y=100 * (m_ * xs + b_), mode='lines', name='Least-squares fit', line=dict(color=GD, width=2), hoverinfo='skip'))
fig.update_layout(title=f'Sharper Combine cuts, more targets (player level r = {r_:.2f}, n = {len(q)}; model: +1.9 pts per SD)', xaxis_title='Combine Cut Force (0 = class average, 1 = one step above)', yaxis_title='Targets per route run (%)', **LAY)
show(fig)

# Appendix H, Table H1: ten break definitions on the same checks
label('Appendix H, Table H1. Ten break definitions on the same checks')
cols = [('reliability', 'Combine<br>reliability', 1), ('r_forty', '|r| with 40<br>(lower = newer)', -1), ('transfer_r', 'Combine to<br>game r', 1),
        ('tprr_r', 'Targets per<br>route r', 1), ('heldout_r', 'Held-out<br>residual r', 1), ('rmse_change', 'Held-out RMSE<br>change (pts)', -1)]
vals = np.array([[abs(v) if c == 'r_forty' else v for c, _, _ in cols for v in [ALT.loc[i, c]]] for i in ALT.index])
z = np.zeros_like(vals)
for j, (_, _, sgn) in enumerate(cols):
    v = vals[:, j] * sgn; z[:, j] = (v - v.min()) / (v.max() - v.min() + 1e-9)
txt = [[f'{vals[i, j]:+.2f}' if cols[j][0] == 'rmse_change' else f'{vals[i, j]:.2f}' for j in range(len(cols))] for i in range(len(ALT))]
fig = go.Figure(go.Heatmap(z=z, x=[c[1] for c in cols], y=ALT.definition, colorscale=[[0, '#ffffff'], [0.12, '#ffffff'], [1, '#9db6d6']], showscale=False,
                           text=txt, texttemplate='%{text}', xgap=3, ygap=3, hovertemplate='%{y}<br>%{x}: %{text}<extra></extra>'))
fig.update_layout(title='Ten break definitions on the same checks (darker is better within a column)', yaxis=dict(autorange='reversed'), height=520, **LAY)
show(fig)

# Appendix H, Table H2: a held-out draft class
label('Appendix H, Table H2. A held-out draft class')
fig = go.Figure()
for m, lab, col in [('peak', 'Peak (primary)', NV), ('first', 'First break', GD)]:
    h = HO[m]
    fig.add_trace(go.Bar(x=[f'{y} (n = {n})' for y, n in zip(h.held_out, h.n)], y=h.resid_r, name=lab, marker_color=col, text=[f'r = {r:.2f}, p = {p:.3f}' for r, p in zip(h.resid_r, h.resid_p)], textposition='outside'))
fig.update_layout(title='Does Cut Force explain what a model fit on the other classes missed?', barmode='group', xaxis_title='Held-out draft class', yaxis=dict(title='r, Cut Force vs held-out residual target rate', range=[-0.2, 0.7], zeroline=True), **LAY)
show(fig)
# Writeup Figure 3: the break-phase scorecard (same numbers and shading as the static figure in section 14)
label('Writeup Figure 3. Each break phase scored on the same checks')
try:
    tdb_ = T10.dropna(subset=['db_cf_z'])
    mdb_ = smf.ols('sep ~ db_cf_z + C(route) + snap_dist', tdb_).fit(cov_type='cluster', cov_kwds={'groups': tdb_.db})
    rows3 = list(PHASES.phase) + ['Defensive back Cut Force']
    V3 = [[r.combine_rel, r.game_rel, r.transfer_r, 1 - r.stopwatch_max_abs_r] for r in PHASES.itertuples()] + [[REL_DB, REL_DBG, r_db, None]]
    T3 = [[f'{r.combine_rel:.2f}', f'{r.game_rel:.2f}', f'{r.transfer_r:+.2f}', f'{r.stopwatch_max_abs_r:.2f}', f'{r.sep_b:+.3f} (p = {r.sep_p:.2f})'] for r in PHASES.itertuples()]
    T3 += [[f'{REL_DB:.2f}', f'{REL_DBG:.2f}', f'{r_db:+.2f}', 'not tested', f'{mdb_.params.db_cf_z:+.2f} (p = {mdb_.pvalues.db_cf_z:.2f})']]
    SCL3 = [(0, 0.7), (0, 0.8), (-0.1, 0.35), (0.4, 1.0)]
    Z3 = [[None if v is None else min(1, max(0, (v - SCL3[j][0]) / (SCL3[j][1] - SCL3[j][0]))) for j, v in enumerate(row)] + [None] for row in V3]
    H3 = ['Combine<br>reliability', 'NFL game<br>reliability', 'Carries to<br>NFL routes (r)', 'Overlap with<br>stopwatch drills<br>(lower = more new info)', 'Separation at<br>throw, yd per SD']
    WHY3 = ['How well one set of Combine drills agrees with another (Spearman-Brown, 1 = perfect)',
            'How well one set of NFL routes agrees with another',
            'Correlation between the Combine score and the same score in NFL routes',
            'Largest correlation with the 40, 10-yard split, 3-cone or shuttle. Lower means newer information',
            'Change in separation at the throw per SD, with weight and slot share controlled. Not shaded']
    CD3 = [[WHY3[j] for j in range(5)] for _ in rows3]
    fig = go.Figure(go.Heatmap(z=Z3, x=H3, y=rows3, text=T3, texttemplate='%{text}', textfont=dict(size=15), customdata=CD3,
                               colorscale=[[0, '#F3F4F6'], [0.5, '#C9D3E3'], [1, NV]], zmin=0, zmax=1, showscale=False, xgap=4, ygap=4, hoverongaps=False,
                               hovertemplate='<b>%{y}</b><br>%{x}: <b>%{text}</b><br>%{customdata}<extra></extra>'))
    fig.add_shape(type='rect', xref='paper', yref='y', x0=-0.355, x1=1.0, y0=0.5, y1=1.5, line=dict(color=GD, width=3))
    fig.update_layout(title='Only the cut itself is a Combine trait that carries to NFL routes', yaxis=dict(autorange='reversed', automargin=True),
                      xaxis=dict(side='top'), height=470, **{**LAY, 'margin': dict(l=230, r=20, t=110, b=30)})
    show(fig)
except Exception as e:
    print('Figure 3 interactive skipped:', repr(e))

# Writeup Figure 6: Cut Force under load (Appendix D)
label('Writeup Figure 6. Cut Force under game load')
try:
    fig = make_subplots(rows=1, cols=2, column_widths=[0.56, 0.44], horizontal_spacing=0.12,
                        subplot_titles=('A. Cut Force holds as game load builds', 'B. Workload numbers track speed, not cuts'))
    y6 = (QL.d / bsd * 100).to_numpy(); e6 = (1.96 * QL.se / bsd * 100).to_numpy()
    fig.add_shape(type='rect', xref='x domain', yref='y', x0=0, x1=1, y0=-10, y1=10, fillcolor='#E3E7EE', opacity=1, line_width=0, layer='below')
    fig.add_trace(go.Scatter(x=QL.cum, y=y6, mode='lines+markers', name='Cut Force vs his own average', line=dict(color=NV, width=2.6), marker=dict(size=10, color=NV),
                             error_y=dict(type='data', array=e6, color=NV, thickness=1.6, width=5), customdata=np.c_[y6 - e6, y6 + e6],
                             hovertemplate='About %{x:.0f} high speed yards already run<br>Cut Force %{y:+.1f}% of a player SD<br>95% interval %{customdata[0]:+.1f} to %{customdata[1]:+.1f}<extra></extra>'), 1, 1)
    L6 = ['High speed yd<br>per route', 'Sprint yd<br>per route', 'Accel + decel<br>efforts per route']
    fig.add_trace(go.Bar(x=L6, y=[abs(WCOR[c][1]) for c in WCOR], name='40 time', marker_color=GD,
                         hovertemplate='%{x}<br>|r| with the 40: %{y:.2f}<extra></extra>'), 1, 2)
    fig.add_trace(go.Bar(x=L6, y=[abs(WCOR[c][0]) for c in WCOR], name='Combine Cut Force', marker_color=NV,
                         hovertemplate='%{x}<br>|r| with Combine Cut Force: %{y:.2f}<extra></extra>'), 1, 2)
    fig.update_xaxes(title_text='High speed yards already run in the game', row=1, col=1)
    fig.update_yaxes(title_text='Cut Force vs own average (% of a player SD)', range=[-40, 40], zeroline=True, row=1, col=1)
    fig.update_yaxes(title_text='|r| with workload metric', range=[0, 0.6], row=1, col=2)
    fig.update_layout(barmode='group', height=470, legend=dict(orientation='h', y=-0.22), **LAY)
    fig.add_annotation(text='shaded band = within 10% of a player SD', xref='x domain', yref='y domain', x=0.98, y=0.04, showarrow=False, font=dict(color='#6B7280', size=12), row=1, col=1)
    show(fig)
except Exception as e:
    print('Figure 6 interactive skipped:', repr(e))

# Writeup Figure 7: the historical Combine (Appendix G). Public nflverse data, so this one needs the internet.
label('Writeup Figure 7. The 2000 to 2026 Combine in the 40 and vertical jump')
NV_URL = 'https://github.com/nflverse/nflverse-data/releases/download/combine/combine.csv'
PG7 = {'QB': 'QB', 'RB': 'RB', 'FB': 'RB', 'WR': 'WR', 'TE': 'TE', 'OT': 'OL', 'OG': 'OL', 'C': 'OL', 'G': 'OL', 'OL': 'OL',
       'DT': 'DL', 'DL': 'DL', 'NT': 'DL', 'DE': 'EDGE', 'EDGE': 'EDGE', 'OLB': 'LB', 'ILB': 'LB', 'LB': 'LB',
       'CB': 'DB', 'S': 'DB', 'SAF': 'DB', 'DB': 'DB', 'CB/WR': 'DB'}
def fig7_frames(nv, cr):
    nv = nv[nv.season != 2021].copy(); nv['grp'] = nv.pos.map(PG7); nv = nv[nv.grp.notna()]
    hist_all = nv[(nv.pos == 'WR') & (nv.season < 2023)]                     # means use every receiver with that measure
    kag_all = cr[cr.combine_position == 'WR'].merge(players[['nfl_id', 'display_name']], on='nfl_id', how='left')
    both = lambda d: d.dropna(subset=['forty', 'vertical'])                     # points need both measures
    nvb = both(nv)
    med = nvb.groupby('grp').agg(forty=('forty', 'median'), vertical=('vertical', 'median'), wt=('wt', 'median'), n=('forty', 'size')).reset_index()
    return nvb, med, both(hist_all), both(kag_all), hist_all, kag_all
try:
    nv7, med7, hist7, kag7, hist7a, kag7a = fig7_frames(pd.read_csv(NV_URL), cr)
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                        subplot_titles=('A. Participants with a 40 and a vertical, by position', 'B. Receivers: this study against 2000 to 2022'))
    fig.add_trace(go.Scattergl(x=nv7.forty, y=nv7.vertical, mode='markers', name=f'Participants with both measures (n = {len(nv7):,})', marker=dict(size=5, color='#C4CAD3', opacity=0.3),
                               text=[f'{n} ({p}, {s})' for n, p, s in zip(nv7.player_name, nv7.pos, nv7.season)], hovertemplate='%{text}<br>40: %{x:.2f} s<br>Vertical: %{y:.1f} in<extra></extra>'), 1, 1)
    fig.add_trace(go.Scatter(x=med7.forty, y=med7.vertical, mode='markers', name='Position median (size = median weight, lb)', text=med7.grp,
                             marker=dict(size=8 + (med7.wt - med7.wt.min()) / (med7.wt.max() - med7.wt.min()) * 22, color=[GD if g == 'WR' else NV for g in med7.grp], line=dict(color='white', width=1.5)),
                             customdata=np.c_[med7.wt, med7.n], hovertemplate='<b>%{text}</b> median<br>40: %{x:.2f} s<br>Vertical: %{y:.1f} in<br>Weight: %{customdata[0]:.0f} lb (n = %{customdata[1]})<extra></extra>'), 1, 1)
    fig.add_trace(go.Scattergl(x=hist7.forty, y=hist7.vertical, mode='markers', name=f'Combine receivers, 2000 to 2022 (n = {len(hist7)})', marker=dict(size=6, color='#B8BEC8', opacity=0.6),
                               text=[f'{n} ({s})' for n, s in zip(hist7.player_name, hist7.season)], hovertemplate='%{text}<br>40: %{x:.2f} s<br>Vertical: %{y:.1f} in<extra></extra>'), 1, 2)
    fig.add_trace(go.Scatter(x=kag7.forty, y=kag7.vertical, mode='markers', name=f'The competition\'s 2023 to 2025 receivers (n = {len(kag7)})', marker=dict(size=8, color=NV, opacity=0.85, line=dict(color='white', width=0.8)),
                             text=[f'{n} ({y})' for n, y in zip(kag7.display_name.fillna('Receiver'), kag7.draft_year)], hovertemplate='%{text}<br>40: %{x:.2f} s<br>Vertical: %{y:.1f} in<extra></extra>'), 1, 2)
    for d7, col, nm in [(hist7a, '#6B7280', '2000 to 2022 mean'), (kag7a, GD, '2023 to 2025 mean')]:
        mx, my = d7.forty.mean(), d7.vertical.mean()
        fig.add_trace(go.Scatter(x=[mx], y=[my], mode='markers', name=nm, marker=dict(symbol='x-thin', size=22, line=dict(color=col, width=4)),
                                 hovertemplate=f'{nm}<br>40: {mx:.2f} s<br>Vertical: {my:.1f} in<extra></extra>'), 1, 2)
    OFF7 = {'WR': (-46, 30), 'DB': (-50, -26), 'RB': (6, -44), 'LB': (34, -34), 'TE': (40, 26), 'QB': (46, 8), 'EDGE': (-6, 40), 'DL': (30, 34), 'OL': (34, 30)}
    for r7 in med7.itertuples():
        ax7, ay7 = OFF7.get(r7.grp, (30, -30))
        fig.add_annotation(x=r7.forty, y=r7.vertical, text=f'<b>{r7.grp}</b>', ax=ax7, ay=ay7, arrowcolor='#9CA3AF', arrowwidth=1, font=dict(size=13, color=GD if r7.grp == 'WR' else NV),
                           bgcolor='rgba(255,255,255,0.85)', row=1, col=1)
    for c in (1, 2):
        fig.update_xaxes(title_text='40-yard dash (s), faster to the left', row=1, col=c)
        fig.update_yaxes(title_text='Vertical jump (in)', row=1, col=c)
    fig.update_xaxes(range=[4.2, 5.6], row=1, col=1); fig.update_xaxes(range=[4.2, 4.85], row=1, col=2)
    fig.update_layout(height=520, legend=dict(orientation='h', y=-0.2), **LAY)
    show(fig)
    print(f'Means over every receiver with that measure. 2000 to 2022: 40 = {hist7a.forty.mean():.2f} s (n = {hist7a.forty.notna().sum()}), vertical = {hist7a.vertical.mean():.1f} in (n = {hist7a.vertical.notna().sum()}) | '
          f'this study: 40 = {kag7a.forty.mean():.2f} s (n = {kag7a.forty.notna().sum()}), vertical = {kag7a.vertical.mean():.1f} in (n = {kag7a.vertical.notna().sum()}). Plotted with both: {len(hist7)} and {len(kag7)}.')
except Exception as e:
    print('Figure 7 interactive skipped (it reads the public nflverse file, so the notebook needs internet on):', repr(e))


## 17. Break replay (Figure 1, animated)

The writeup's Figure 1 is an animated GIF built here from the same two Combine speed-out reps and saved to `/kaggle/working/fig1_cut_force.gif`. Its last frame is the static Figure 1 above. The interactive version below has a Play button and a time slider. The arrow is the sideways (centripetal) acceleration at each instant, in g, drawn to scale at 3.6 yards per g. It points to the inside of the turn from the signed change in heading.

In [ ]:
import io
from PIL import Image

def replay_data(d):
    """One rep turned the same way as Figure 1: upfield along x, the cut toward +y."""
    x, y = d.x.values - d.x.values[0], d.y.values - d.y.values[0]
    spd = smooth(d.s.values)
    lat = spd * np.radians(np.abs(np.gradient(smooth(np.degrees(np.unwrap(np.radians(d.dir.values))))) * 10)) / G
    k = int(np.argmax(np.where((np.arange(len(lat)) > 8) & (spd > 3), lat, -1))); j = max(5, k - 4)
    ang = np.arctan2(y[j], x[j]); x, y = np.cos(-ang) * x - np.sin(-ang) * y, np.sin(-ang) * x + np.cos(-ang) * y
    if y[min(len(y) - 1, k + 6)] < 0:
        y = -y
    tx, ty = np.gradient(x), np.gradient(y); nrm = np.hypot(tx, ty) + 1e-9
    turn = np.sign(np.gradient(np.unwrap(np.arctan2(ty, tx)))); turn[turn == 0] = 1
    return dict(x=x, y=y, lat=lat, k=k, t=np.arange(len(lat)) / 10, tx=tx, ty=ty, nx=-ty / nrm * turn, ny=tx / nrm * turn)

hi, lo = so.iloc[int(len(so) * .9)], so.iloc[int(len(so) * .1)]
RP2 = [dict(replay_data(REPS[(r.nfl_id, r.event_id, r.attempt, 'SPEED_OUT_ROUTE_LEFT')]), col=c, lab=f'{lab} ({r.g:.2f} g)')
       for r, c, lab in [(hi, NAVY, 'High Cut Force'), (lo, GOLD, 'Low Cut Force')]]

def replay_gif(R, out, width_px=1600, fps=15, speed=0.6, hold=5.0, yd_per_g=3.6):
    """Animated Figure 1. Same styling as the static figure; the last frame is the static figure, held for five seconds."""
    with plt.rc_context({'font.family': 'DejaVu Sans', 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'}):
        T = max(p['t'][-1] for p in R)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
        art = []
        for p in R:
            ax[0].plot(p['x'], p['y'], color=p['col'], lw=2.6, alpha=0.13)
            a = dict(ln=ax[0].plot([], [], color=p['col'], lw=2.6, label=p['lab'])[0],
                     pk=ax[0].scatter([p['x'][p['k']]], [p['y'][p['k']]], s=70, color=p['col'], zorder=5, ec='white'),
                     run=ax[0].plot([], [], 'o', ms=10, mfc='white', mec=p['col'], mew=2.4, zorder=6)[0],
                     arr=ax[0].annotate('', xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=p['col'], lw=2.6, mutation_scale=16), zorder=7),
                     ln2=ax[1].plot([], [], color=p['col'], lw=2.4)[0],
                     pk2=ax[1].scatter([p['t'][p['k']]], [p['lat'][p['k']]], s=50, color=p['col'], zorder=5))
            art.append(a)
        ax[0].set_xlabel('Yards upfield'); ax[0].set_ylabel('Yards across'); ax[0].set_title('Route path, same Combine speed out')
        ax[0].legend(frameon=False, loc='upper left'); panel(ax[0], 'A')
        ax[0].text(0.98, 0.04, 'dots mark the peak cut', transform=ax[0].transAxes, ha='right', fontsize=11, color=GRAY)
        note = ax[0].text(0.02, 0.70, 'arrow = lateral force (g)\nat that instant', transform=ax[0].transAxes, ha='left', va='top', fontsize=11, color=GRAY)
        scale = [ax[0].annotate('', xy=(6.5, 2.4 + 0.5 * yd_per_g), xytext=(6.5, 2.4), arrowprops=dict(arrowstyle='-|>', color=GRAY, lw=2, mutation_scale=13)),
                 ax[0].text(6.85, 2.4 + 0.25 * yd_per_g, 'scale: 0.5 g', va='center', fontsize=11, color=GRAY)]
        ax[1].set_xlabel('Seconds into rep'); ax[1].set_ylabel('Lateral force (g)'); ax[1].set_title('Lateral force over the rep'); panel(ax[1], 'B')
        cur = ax[1].axvline(0, color='#9CA3AF', lw=1.2, ls='--', zorder=1)
        clock = ax[1].text(0.98, 0.95, '', transform=ax[1].transAxes, ha='right', va='top', fontsize=12, color=GRAY, fontweight='bold')
        allx = np.concatenate([p['x'] for p in R]); ally = np.concatenate([p['y'] for p in R])
        ax[0].set_xlim(allx.min() - 0.9, allx.max() + 0.9); ax[0].set_ylim(ally.min() - 0.9, ally.max() + 0.6)
        ax[1].set_xlim(-0.25, T + 0.25); ax[1].set_ylim(-0.035, max(p['lat'].max() for p in R) * 1.06)
        fig.tight_layout(); fig.canvas.draw()
        to_px, to_data = ax[0].transData.transform, ax[0].transData.inverted().transform
        ypx = abs(to_px((0, 1))[1] - to_px((0, 0))[1])
        frames = []
        for t in list(np.arange(0, T + 1e-9, speed / fps)) + [None]:
            final = t is None
            for p, a in zip(R, art):
                n = len(p['t']); tt = p['t'][-1] if final else min(t, p['t'][-1])
                f = min(tt * 10, n - 1); i = int(f); w = f - i; j = min(i + 1, n - 1)
                pos = np.array([p['x'][i] * (1 - w) + p['x'][j] * w, p['y'][i] * (1 - w) + p['y'][j] * w])
                tan = np.array([p['tx'][i] * (1 - w) + p['tx'][j] * w, p['ty'][i] * (1 - w) + p['ty'][j] * w])
                g = p['lat'][i] * (1 - w) + p['lat'][j] * w
                sgn = np.sign(p['nx'][i] * -tan[1] + p['ny'][i] * tan[0]) or 1.0           # same side as the turn
                a['ln'].set_data(np.append(p['x'][:i + 1], pos[0]), np.append(p['y'][:i + 1], pos[1]))
                a['ln2'].set_data(np.append(p['t'][:i + 1], tt), np.append(p['lat'][:i + 1], g))
                a['pk'].set_visible(final or tt >= p['t'][p['k']]); a['pk2'].set_visible(final or tt >= p['t'][p['k']])
                moving = (not final) and t <= p['t'][-1]
                a['run'].set_data([pos[0]] if moving else [], [pos[1]] if moving else [])
                if moving and g > 0.04:
                    P0, P1 = to_px(pos), to_px(pos + tan / (np.hypot(*tan) + 1e-9))
                    u = (P1 - P0) / (np.hypot(*(P1 - P0)) + 1e-9)
                    a['arr'].xy = tuple(to_data(P0 + np.array([-u[1], u[0]]) * sgn * g * yd_per_g * ypx)); a['arr'].set_position(tuple(pos)); a['arr'].set_visible(True)
                else:
                    a['arr'].set_visible(False)
            cur.set_visible(not final); note.set_visible(not final)
            for sc in scale:
                sc.set_visible(not final)
            if final:
                clock.set_text('')
            else:
                cur.set_xdata([t, t]); clock.set_text(f'{t:.1f} s (0.6x speed)')
            buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=width_px / 12); buf.seek(0)
            frames.append(Image.open(buf).convert('RGB'))
        plt.close(fig)
    keys = ['#FFFFFF', NAVY, GOLD, GRAY, '#1A1A1A', '#000000', '#E5E7EB', '#9CA3AF']
    cols = [tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) for h in keys]
    mc = frames[len(frames) // 2].quantize(colors=200, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE).getpalette()[:600]
    for i in range(0, len(mc), 3):
        c = tuple(mc[i:i + 3])
        if all(tuple(v >> 3 for v in c) != tuple(v >> 3 for v in k) for k in cols):     # one color per 5-bit bucket keeps white white
            cols.append(c)
    pal = Image.new('P', (1, 1)); pal.putpalette([v for c in cols[:256] for v in c] + [0] * (768 - 3 * len(cols[:256])))
    q = [f.quantize(palette=pal, dither=Image.Dither.NONE) for f in frames]
    q[0].save(out, save_all=True, append_images=q[1:], duration=[round(1000 / fps)] * (len(q) - 1) + [int(hold * 1000)], loop=0, disposal=1)
    return len(q)

os.makedirs('/kaggle/working', exist_ok=True) if os.path.isdir('/kaggle') else None
GIF_OUT = '/kaggle/working/fig1_cut_force.gif' if os.path.isdir('/kaggle/working') else 'fig1_cut_force.gif'
nf = replay_gif(RP2, GIF_OUT)
print(f'{GIF_OUT}: {nf} frames, {os.path.getsize(GIF_OUT) / 1e6:.1f} MB. Peaks {RP2[0]["lat"][RP2[0]["k"]]:.2f} g and {RP2[1]["lat"][RP2[1]["k"]]:.2f} g')
print('Legend, Cut Force of each rep:', ' | '.join(p['lab'] for p in RP2))

def replay_plotly(R, yd_per_g=3.6):
    n = max(len(p['t']) for p in R)
    def state(i):
        tr = []
        for p in R:
            m = min(i, len(p['t']) - 1); live = i < len(p['t']) - 1; g = p['lat'][m]; done = m >= p['k']   # no runner once the rep is over
            tip = (p['x'][m] + p['nx'][m] * g * yd_per_g, p['y'][m] + p['ny'][m] * g * yd_per_g)
            tr += [go.Scatter(x=p['x'][:m + 1], y=p['y'][:m + 1]),
                   go.Scatter(x=[p['x'][m]] if live else [], y=[p['y'][m]] if live else []),
                   go.Scatter(x=[p['x'][m], tip[0]] if live and g > 0.04 else [], y=[p['y'][m], tip[1]] if live and g > 0.04 else []),
                   go.Scatter(x=[p['x'][p['k']]] if done else [], y=[p['y'][p['k']]] if done else []),
                   go.Scatter(x=p['t'][:m + 1], y=p['lat'][:m + 1]),
                   go.Scatter(x=[p['t'][p['k']]] if done else [], y=[p['lat'][p['k']]] if done else [])]
        t = min(i, n - 1) / 10
        return tr + [go.Scatter(x=[t, t], y=[0, 10])]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.56, 0.44], horizontal_spacing=0.09,
                        subplot_titles=('Route path, same Combine speed out', 'Lateral force over the rep'))
    for p in R:
        fig.add_trace(go.Scatter(x=p['x'], y=p['y'], mode='lines', line=dict(color=p['col'], width=2.5), opacity=0.15, hoverinfo='skip', showlegend=False), 1, 1)
    base = state(n - 1)                                                       # opens on the finished rep, like the static figure
    for p, j in zip(R, (0, 6)):
        c = p['col']
        fig.add_trace(go.Scatter(x=base[j].x, y=base[j].y, mode='lines', line=dict(color=c, width=3), name=p['lab'], hoverinfo='skip'), 1, 1)
        fig.add_trace(go.Scatter(x=base[j + 1].x, y=base[j + 1].y, mode='markers', marker=dict(size=13, color='white', line=dict(color=c, width=3)), hoverinfo='skip', showlegend=False), 1, 1)
        fig.add_trace(go.Scatter(x=base[j + 2].x, y=base[j + 2].y, mode='lines+markers', line=dict(color=c, width=3),
                                 marker=dict(symbol='arrow', size=[0, 15], angleref='previous', color=c), hoverinfo='skip', showlegend=False), 1, 1)
        fig.add_trace(go.Scatter(x=base[j + 3].x, y=base[j + 3].y, mode='markers', marker=dict(size=11, color=c, line=dict(color='white', width=1.5)),
                                 hovertemplate='Peak cut<extra></extra>', showlegend=False), 1, 1)
        fig.add_trace(go.Scatter(x=base[j + 4].x, y=base[j + 4].y, mode='lines', line=dict(color=c, width=2.6), showlegend=False,
                                 hovertemplate='%{x:.1f} s: %{y:.2f} g<extra></extra>'), 1, 2)
        fig.add_trace(go.Scatter(x=base[j + 5].x, y=base[j + 5].y, mode='markers', marker=dict(size=10, color=c),
                                 hovertemplate='Peak: %{y:.2f} g at %{x:.1f} s<extra></extra>', showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=base[12].x, y=base[12].y, mode='lines', line=dict(color='#9CA3AF', width=1.2, dash='dash'), hoverinfo='skip', showlegend=False), 1, 2)
    fig.frames = [go.Frame(data=state(i), traces=list(range(2, 15)), name=f'{i / 10:.1f}') for i in range(n)]
    fig.add_annotation(x=6.5, y=2.4 + 0.5 * yd_per_g, ax=6.5, ay=2.4, xref='x', yref='y', axref='x', ayref='y', showarrow=True, arrowhead=2,
                       arrowwidth=2, arrowcolor='#6B7280', text='')
    fig.add_annotation(x=6.9, y=2.4 + 0.25 * yd_per_g, xref='x', yref='y', text='scale: 0.5 g', showarrow=False, xanchor='left', font=dict(color='#6B7280', size=12))
    ax_, ay_ = np.concatenate([p['x'] for p in R]), np.concatenate([p['y'] for p in R])
    fig.update_xaxes(title_text='Yards upfield', range=[ax_.min() - 1, ax_.max() + 1], constrain='domain', row=1, col=1)
    fig.update_yaxes(title_text='Yards across', range=[ay_.min() - 1.5, ay_.max() + 1], scaleanchor='x', scaleratio=1, row=1, col=1)
    fig.update_xaxes(title_text='Seconds into rep', range=[-0.15, (n - 1) / 10 + 0.15], row=1, col=2)
    fig.update_yaxes(title_text='Lateral force (g)', range=[-0.03, max(p['lat'].max() for p in R) * 1.1], row=1, col=2)
    play = dict(label='Play', method='animate', args=[None, dict(frame=dict(duration=140, redraw=True), transition=dict(duration=0), fromcurrent=False, mode='immediate')])
    pause = dict(label='Pause', method='animate', args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
    steps = [dict(method='animate', label=f.name, args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')]) for f in fig.frames]
    fig.update_layout(height=480, **{**LAY, 'margin': dict(l=60, r=20, t=70, b=95)}, legend=dict(x=0.08, y=0.97, bgcolor='rgba(255,255,255,0.7)'),
                      title=dict(text='Break replay: the arrow is the lateral force at each instant, in g (press Play)', y=0.97),
                      updatemenus=[dict(type='buttons', direction='left', x=0, y=-0.17, xanchor='left', yanchor='top', buttons=[play, pause], pad=dict(r=8, t=0))],
                      sliders=[dict(active=n - 1, x=0.15, len=0.85, y=-0.13, yanchor='top', steps=steps, pad=dict(t=0), ticklen=0, minorticklen=0,
                                    font=dict(color='rgba(0,0,0,0)'), currentvalue=dict(prefix='t = ', suffix=' s', xanchor='right', font=dict(color='#374151', size=13)))])
    return fig

show(replay_plotly(RP2))

## 18. Cut Force explorer

Every receiver in the Combine sample on one chart. The menus switch the comparison between draft pick, 40-yard dash, and 3-cone, and filter by draft class. Color is NFL targets per route run for receivers with at least 100 regular-season routes, size is routes run, and hollow markers have fewer than 100 routes so far and carry no target-rate color. Hover for the receiver.

In [ ]:
SCALE = [[0.0, '#C8CFD9'], [0.45, '#D9B44A'], [0.75, '#B08400'], [1.0, '#13294B']]   # light to navy, every step visible on white

def explorer(P, rep_g):
    Q = P.copy()
    Q['cf_g'] = Q.nfl_id.map(rep_g); Q['tp'] = 100 * Q.tprr; Q['pct'] = 100 * Q.cf_z.rank(pct=True)
    Q['gz'] = (Q.game_cf - Q.game_cf.mean()) / Q.game_cf.std()
    Q['pk'] = np.where(Q.draft_overall_pick.notna(), 'pick ' + Q.draft_overall_pick.fillna(0).astype(int).astype(str), 'undrafted')
    Q['qual'] = Q.rr.fillna(0) >= 100
    f2 = lambda v: 'n/a' if pd.isna(v) else f'{v:.2f}'
    Q['txt'] = [f'<b>{r.display_name}</b><br>{int(r.draft_year)} class, {r.pk}'
                f'<br>Combine Cut Force {r.cf_z:+.2f} SD ({r.pct:.0f}th percentile), {r.cf_g:.2f} g per rep'
                f'<br>40: {f2(r.forty)} s, 3-cone: {f2(r.three_cone)} s'
                + (f'<br>NFL: {int(r.rr):,} routes, {r.tp:.1f}% targeted' if r.rr == r.rr else '<br>NFL: no regular-season routes yet')
                + (f'<br>Game Cut Force {r.gz:+.2f} SD' if r.gz == r.gz else '') for r in Q.itertuples()]
    XS = [('pick', 'Draft pick (log scale)', 'log', 'Up and right: later pick, sharper cuts', ''),
          ('forty', '40-yard dash (s)', 'linear', 'Up and right: slower 40, sharper cuts', ' Receivers without a Combine 40 are not shown in this view.'),
          ('three_cone', '3-cone (s)', 'linear', 'Up and right: slower 3-cone, sharper cuts', ' Receivers without a Combine 3-cone are not shown in this view.')]
    lo, hi = np.nanpercentile(Q.loc[Q.qual, 'tp'], [5, 95])
    fig = go.Figure(); groups = []
    for yr in sorted(Q.draft_year.unique()):
        for qual in (True, False):
            d = Q[(Q.draft_year == yr) & (Q.qual == qual)]
            mk = (dict(size=np.clip(np.sqrt(d.rr) / 2.2, 8, 24), color=d.tp, coloraxis='coloraxis', line=dict(color='#4B5563', width=0.6))
                  if qual else dict(size=9, color='white', line=dict(color='#9CA3AF', width=1.6)))
            fig.add_trace(go.Scatter(x=d['pick'], y=d.cf_z, mode='markers', marker=mk, text=d.txt, hovertemplate='%{text}<extra></extra>', showlegend=False))
            groups.append((yr, d))
    fig.add_hline(y=0, line=dict(color='#9CA3AF', width=1, dash='dot'))
    def ann(note, extra):
        return [dict(text=note, xref='x domain', yref='y domain', x=0.99, y=0.99, showarrow=False, xanchor='right', yanchor='top', font=dict(color='#6B7280', size=12)),
                dict(text='Hollow = under 100 NFL routes so far. Size = routes run.' + extra, xref='paper', yref='paper', x=0, y=-0.19,
                     showarrow=False, xanchor='left', font=dict(color='#6B7280', size=11.5)),
                dict(text='Draft class', xref='paper', yref='paper', x=0, y=1.25, showarrow=False, xanchor='left', yanchor='bottom', font=dict(size=12, color='#374151')),
                dict(text='Compare Cut Force with', xref='paper', yref='paper', x=0.22, y=1.25, showarrow=False, xanchor='left', yanchor='bottom', font=dict(size=12, color='#374151'))]
    TICK = dict(tickvals=[1, 3, 10, 32, 64, 100, 160, 262], ticktext=['1', '3', '10', '32', '64', '100', '160', 'UDFA'])
    cls_btn = [dict(label='All classes', method='restyle', args=[{'visible': [True] * len(groups)}])] + \
              [dict(label=f'{y} class', method='restyle', args=[{'visible': [g[0] == y for g in groups]}]) for y in sorted(Q.draft_year.unique())]
    x_btn = [dict(label=lab.split(' (')[0], method='update',
                  args=[{'x': [g[1][col] for g in groups]},
                        {'xaxis.title.text': lab, 'xaxis.type': typ, 'xaxis.autorange': True, 'annotations': ann(note, extra),
                         'xaxis.tickvals': TICK['tickvals'] if col == 'pick' else None, 'xaxis.ticktext': TICK['ticktext'] if col == 'pick' else None}])
             for col, lab, typ, note, extra in XS]
    fig.update_layout(height=600, **{**LAY, 'margin': dict(l=70, r=20, t=165, b=90)},
                      title=dict(text='Cut Force explorer: every receiver in the sample (hover for the player)', y=0.98),
                      coloraxis=dict(colorscale=SCALE, cmin=lo, cmax=hi, colorbar=dict(title='NFL targets<br>per route (%)', thickness=14)),
                      xaxis=dict(title='Draft pick (log scale)', type='log', **TICK), yaxis=dict(title='Combine Cut Force (0 = class average, 1 = one step above)', zeroline=False),
                      annotations=ann(XS[0][3], XS[0][4]),
                      updatemenus=[dict(buttons=cls_btn, x=0, y=1.08, xanchor='left', yanchor='bottom', showactive=True, active=0),
                                   dict(buttons=x_btn, x=0.22, y=1.08, xanchor='left', yanchor='bottom', showactive=True, active=0)])
    return fig

show(explorer(P, reps.groupby('nfl_id').g.mean()))

## 19. NFL route maps: where each break happens

Seven receivers with at least 300 regular-season breaking routes: the four highest and the three lowest Combine Cut Force scores. Each line is one route from the snap to the pass, turned so upfield runs to the right and the middle of the field is up. Each dot is the break, colored by its raw Cut Force in g on one color scale for all seven receivers. The title's NFL game Cut Force is the adjusted score from section 4, so the dots and the title are not the same quantity. Choose a receiver from the menu, and click a route type in the legend to hide or show it. Up to 150 routes per receiver are drawn (seed 2027). The counts and the game score use every route. The maps describe selected receivers. The test of whether Combine Cut Force carries over is in section 4.

In [ ]:
nbrk = GF.groupby('nfl_id').size()
Q19 = P[P.nfl_id.isin(nbrk[nbrk >= 300].index)].sort_values('cf_z', ascending=False)
SEL = pd.concat([Q19.head(4), Q19.tail(3)])
ids = set(SEL.nfl_id); keys = GF.loc[GF.nfl_id.isin(ids), ['game_id', 'play_id', 'nfl_id', 'route']]
parts = []
for f in sorted(glob.glob(f'{DATA}/game_tracking_*.csv')):
    for ch in pd.read_csv(f, usecols=['game_id', 'play_id', 'nfl_id', 'time', 'x', 'y', 's', 'dir', 'event'], chunksize=4_000_000):
        parts.append(ch[ch.nfl_id.isin(ids)])
FR = pd.concat(parts).merge(keys, on=['game_id', 'play_id', 'nfl_id']).sort_values(['game_id', 'play_id', 'nfl_id', 'time'])

def route_paths(FR, keep=150, seed=SEED):
    """Snap to pass, the same window as section 4. Upfield is +x and the middle of the field is +y."""
    out = []
    for (g, p, n, r), d in FR.groupby(['game_id', 'play_id', 'nfl_id', 'route'], sort=False):
        ev = d.event.to_numpy(); snap = np.where(ev == 'ball_snap')[0]
        if len(snap) == 0:
            continue
        st = snap[0]; pf = np.where(np.isin(ev, ['pass_forward', 'qb_sack', 'qb_strip_sack']))[0]; pf = pf[pf > st]
        en = pf[0] + 1 if len(pf) else st + 35
        b = cut_force(d.s.to_numpy()[st:en], d.dir.to_numpy()[st:en])
        if not b:
            continue
        x, y = d.x.to_numpy()[st:en], d.y.to_numpy()[st:en]
        sx = 1.0 if x[-1] >= x[0] else -1.0; sy = 1.0 if y[0] < 53.3 / 2 else -1.0
        out.append(dict(game_id=g, play_id=p, nfl_id=n, route=r, g=b['cut_force'] / G, entry=b['entry'], k=b['k'],
                        X=np.round((x - x[0]) * sx, 2), Y=np.round((y - y[0]) * sy, 2)))
    RT = pd.DataFrame(out); RT['shown'] = False; r_ = np.random.default_rng(seed)
    for n, d in RT.groupby('nfl_id'):
        RT.loc[r_.choice(d.index, min(keep, len(d)), replace=False), 'shown'] = True
    return RT

def route_maps(RT, SEL, P, games):
    wk = games.set_index('game_id')[['season', 'week']]
    lo, hi = np.percentile(RT.g, [5, 95]); gz = (P.set_index('nfl_id').game_cf - P.game_cf.mean()) / P.game_cf.std()
    order = ['SLANT', 'OUT', 'IN', 'HITCH', 'CROSS', 'CORNER', 'POST']
    fig = go.Figure(); owner, titles = [], []
    for i, s in enumerate(SEL.itertuples()):
        D = RT[(RT.nfl_id == s.nfl_id) & RT.shown]
        for rt in [o for o in order if o in set(D.route)]:
            d = D[D.route == rt]; lx, ly = [], []
            for r in d.itertuples():
                lx += list(r.X) + [None]; ly += list(r.Y) + [None]
            sw = wk.reindex(d.game_id)
            txt = [f'{rt} route, {int(a)} week {int(b)}<br>Cut Force at the break: {r.g:.2f} g<br>Entry speed {r.entry * 2.045:.1f} mph'
                   for r, a, b in zip(d.itertuples(), sw.season, sw.week)]
            grp = f'{s.nfl_id}-{rt}'
            fig.add_trace(go.Scatter(x=lx, y=ly, mode='lines', line=dict(color=NV, width=1.2), opacity=0.25, hoverinfo='skip', visible=i == 0,
                                     name=f'{rt.title()} ({len(d)})', legendgroup=grp))
            fig.add_trace(go.Scatter(x=[r.X[r.k] for r in d.itertuples()], y=[r.Y[r.k] for r in d.itertuples()], mode='markers', visible=i == 0,
                                     marker=dict(size=8, color=d.g, coloraxis='coloraxis', line=dict(color='#374151', width=0.5)),
                                     text=txt, hovertemplate='%{text}<extra></extra>', legendgroup=grp, showlegend=False))
            owner += [i, i]
        allr = RT[RT.nfl_id == s.nfl_id]
        pk = f'pick {int(s.draft_overall_pick)}' if s.draft_overall_pick == s.draft_overall_pick else 'undrafted'
        titles.append(f'{s.display_name}: Combine Cut Force {s.cf_z:+.1f} SD, NFL game Cut Force {gz.get(s.nfl_id, np.nan):+.1f} SD (adjusted)'
                      f'<br><sup>{int(s.draft_year)} class, {pk}. {len(allr):,} regular-season breaking routes, {int(allr.shown.sum())} drawn.</sup>'
                      f'<br><sup>Dots are raw Cut Force at the break. Descriptive only: the transfer test is section 4.</sup>')
    btn = [dict(label=s.display_name, method='update', args=[{'visible': [o == i for o in owner]}, {'title.text': titles[i]}]) for i, s in enumerate(SEL.itertuples())]
    shapes = [dict(type='line', x0=v, x1=v, y0=-40, y1=60, line=dict(color='white', width=2 if v % 10 == 0 else 1), layer='below') for v in range(-10, 60, 5)]
    shapes.append(dict(type='line', x0=0, x1=0, y0=-40, y1=60, line=dict(color=NV, width=2, dash='dash'), layer='below'))
    fig.update_layout(height=720, **{**LAY, 'margin': dict(l=70, r=20, t=135, b=95)}, plot_bgcolor='#E7F0E3', shapes=shapes,
                      title=dict(text=titles[0], y=0.97), coloraxis=dict(colorscale=SCALE, cmin=lo, cmax=hi, colorbar=dict(title='Raw Cut Force<br>at the break (g)', thickness=14)),
                      legend=dict(title='Route type (click to hide):', orientation='h', x=0, y=-0.12, font=dict(size=12)),
                      xaxis=dict(title='Yards upfield from the snap (dashed line = line of scrimmage)', range=[-4, 30], constrain='domain', showgrid=False, zeroline=False),
                      yaxis=dict(title='Yards toward the middle of the field', range=[-14, 22], scaleanchor='x', scaleratio=1, showgrid=False, zeroline=False),
                      annotations=[dict(text='toward the middle', x=29.5, y=21.5, xanchor='right', yanchor='top', showarrow=False, font=dict(color='#4B5563', size=11)),
                                   dict(text='toward the sideline', x=29.5, y=-13.5, xanchor='right', yanchor='bottom', showarrow=False, font=dict(color='#4B5563', size=11))],
                      updatemenus=[dict(buttons=btn, x=0, y=1.03, xanchor='left', yanchor='bottom', showactive=True, active=0)])
    return fig

RT = route_paths(FR)
print(f'{len(RT):,} breaking routes for {RT.nfl_id.nunique()} receivers; route Cut Force 5th, 50th, 95th pct:', np.percentile(RT.g, [5, 50, 95]).round(2).tolist())
show(route_maps(RT, SEL, P, games))

## 20. Every receiver: his card and his highest Cut Force rep

The writeup shows one Cut Force card (Figure 5, Puka Nacua) and two reps (Figure 1). Here all 107 receivers get both. Pick a name from the menu above each chart.

The card has the same three panels as Figure 5: his percentiles against the class, every Combine route drill against the middle 80% of the class, and what happened in the NFL. The replay shows the one Combine rep with his highest Cut Force, turned the same way as Figure 1, with the sideways (centripetal) acceleration across the rep. The dot marks the peak frame. A rep's Cut Force averages the top three frames around that peak, so it can sit just under the dot.

In [ ]:
# Writeup Figure 5 for every receiver: the Cut Force card, chosen from a menu
NAMES20 = P.sort_values('cf_z', ascending=False).display_name.tolist()     # menu order: highest Cut Force first
DR20 = sorted(reps.drill.unique()); NICE20 = [d.replace('_ROUTE', '').replace('_', ' ').title() for d in DR20]
CLS20 = {d: reps[reps.drill == d].res for d in DR20}
q100 = P[P.rr >= 100]
def pct20(col, v, low_good=False):
    s = P[col].dropna()
    return None if pd.isna(v) else float(((s > v) if low_good else (s < v)).mean() * 100)
ITEMS20 = ['Cut Force', 'NGS athleticism', '40-yard dash', '3-cone', 'Draft slot (earlier = higher)']
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.2,
                    subplot_titles=('What the stopwatch saw vs the break (percentile)', 'Every Combine route drill (z vs class)'))
fig.add_trace(go.Bar(y=ITEMS20[::-1], x=[100] * 5, orientation='h', marker_color='#E9ECF1', hoverinfo='skip', showlegend=False, width=0.6), 1, 1)
fig.update_layout(barmode='overlay')
for i, d in enumerate(DR20):
    lo_, hi_, md_ = CLS20[d].quantile(.1), CLS20[d].quantile(.9), CLS20[d].median()
    fig.add_trace(go.Scatter(x=[lo_, hi_], y=[NICE20[i]] * 2, mode='lines', line=dict(color='#D9DEE7', width=12), hoverinfo='skip', showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[md_], y=[NICE20[i]], mode='markers', marker=dict(symbol='line-ns-open', size=16, color='#6B7280', line=dict(width=2)),
                             hovertemplate=f'{NICE20[i]}<br>class median {md_:+.2f}, middle 80% {lo_:+.2f} to {hi_:+.2f}<extra></extra>', showlegend=False), 1, 2)
NBASE20 = len(fig.data); ann20 = {}
for k, nm in enumerate(NAMES20):
    p = P.set_index('display_name').loc[nm]
    v = [pct20('cf_z', p.cf_z), pct20('ngs_athleticism_score', p.ngs_athleticism_score), pct20('forty', p.forty, True), pct20('three_cone', p.three_cone, True), pct20('lpick', p.lpick, True)]
    xs = [0 if a is None else a for a in v][::-1]
    tx = ['did not run' if a is None else f'{a:.0f}' for a in v][::-1]
    fig.add_trace(go.Bar(y=ITEMS20[::-1], x=xs, orientation='h', width=0.6, marker_color=[NV, NV, NV, NV, GD], text=tx, textposition='outside', cliponaxis=False,
                         hovertemplate='%{y}: %{text}<extra></extra>', visible=(k == 0), showlegend=False), 1, 1)
    mine = reps[reps.nfl_id == p.nfl_id].groupby('drill').res.max().reindex(DR20)
    fig.add_trace(go.Scatter(x=mine.values, y=NICE20, mode='markers', marker=dict(size=12, color=GD, line=dict(color=NV, width=1)),
                             hovertemplate='%{y}<br>his best rep: %{x:+.2f}<extra></extra>', visible=(k == 0), showlegend=False), 1, 2)
    g_pct = f'{(P.game_cf.dropna() <= p.game_cf).mean() * 100:.0f}th pct' if pd.notna(p.game_cf) else 'n/a'
    tp = f'{p.tprr * 100:.1f}%' if pd.notna(p.tprr) else 'n/a'
    rr = f'{int(p.rr):,}' if pd.notna(p.rr) else '0'
    ann20[nm] = [dict(text=f'<b>{nm}</b>   WR   {int(p.draft_year)} draft, pick {int(p.pick)}<br><span style="font-size:12px;color:#6B7280">Percentiles among the {len(P)} receivers with Combine route-drill tracking (2023 to 2025 classes)</span>',
                      x=0, y=1.27, xref='paper', yref='paper', showarrow=False, xanchor='left', align='left', font=dict(size=19, color=NV)),
                 dict(text=f'<b>What happened in the NFL</b><br>Targets per route: <b>{tp}</b> (median of receivers with 100+ routes {q100.tprr.median() * 100:.1f}%)'
                           f'<br>NFL game Cut Force: <b>{g_pct}</b> (route and speed adjusted)<br>Regular-season routes run: <b>{rr}</b> (2023 to 2025)',
                      x=0, y=-0.3, xref='paper', yref='paper', showarrow=False, xanchor='left', align='left', font=dict(size=13, color='#1A1A1A'),
                      bgcolor='#F3F4F6', borderpad=8)]
fig.add_annotation(text='gold = his best rep, bar = middle 80%, tick = median', xref='x2 domain', yref='y2 domain', x=1.0, y=-0.12,
                   showarrow=False, xanchor='right', font=dict(size=11, color='#6B7280'))
STATIC20 = list(fig.layout.annotations)          # the two panel titles and the key stay on every receiver
btn20 = []
for k, nm in enumerate(NAMES20):
    vis = [True] * NBASE20 + [False] * (2 * len(NAMES20)); vis[NBASE20 + 2 * k] = vis[NBASE20 + 2 * k + 1] = True
    btn20.append(dict(label=nm, method='update', args=[{'visible': vis}, {'annotations': STATIC20 + ann20[nm]}]))
start = NAMES20.index('Puka Nacua') if 'Puka Nacua' in NAMES20 else 0
for j, t in enumerate(fig.data[NBASE20:]):
    t.visible = (j // 2 == start)
fig.update_layout(updatemenus=[dict(buttons=btn20, active=start, x=1.0, xanchor='right', y=1.32, yanchor='top', direction='down', bgcolor='white', font=dict(size=13))],
                  annotations=STATIC20 + ann20[NAMES20[start]], height=640, **{**LAY, 'margin': dict(l=200, r=30, t=150, b=170)})
fig.update_xaxes(range=[0, 118], tickvals=[0, 50, 100], row=1, col=1)
fig.update_xaxes(zeroline=True, zerolinecolor='#9CA3AF', row=1, col=2)
fig.update_yaxes(autorange='reversed', row=1, col=2)
show(fig)


In [ ]:
# Writeup Figure 1 for every receiver: his highest Cut Force rep, chosen from a menu
best20 = reps.loc[reps.groupby('nfl_id').g.idxmax()].merge(P[['nfl_id', 'display_name', 'cf_z']], on='nfl_id').sort_values('cf_z', ascending=False)
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.1,
                    subplot_titles=('Route path (upfield to the right)', 'Sideways acceleration across the rep (g)'))
B20 = []
for k, r in enumerate(best20.itertuples()):
    R = replay_data(REPS[(r.nfl_id, r.event_id, r.attempt, r.drill)]); kk = R['k']; vis = (k == 0)
    R = {**R, 'x': np.round(R['x'], 2), 'y': np.round(R['y'], 2), 'lat': np.round(R['lat'], 3), 't': np.round(R['t'], 1)}
    drill = r.drill.replace('_ROUTE', '').replace('_', ' ').title()
    fig.add_trace(go.Scatter(x=R['x'], y=R['y'], mode='lines', line=dict(color=NV, width=3), visible=vis, showlegend=False,
                             customdata=np.c_[R['t'], R['lat']], hovertemplate='t = %{customdata[0]:.1f} s<br>%{customdata[1]:.2f} g<extra></extra>'), 1, 1)
    fig.add_trace(go.Scatter(x=[R['x'][kk]], y=[R['y'][kk]], mode='markers', marker=dict(size=13, color=GD, line=dict(color='white', width=1.5)), visible=vis, showlegend=False,
                             hovertemplate=f'peak frame: {R["lat"][kk]:.2f} g<extra></extra>'), 1, 1)
    fig.add_trace(go.Scatter(x=R['t'], y=R['lat'], mode='lines', line=dict(color=NV, width=2.6), visible=vis, showlegend=False,
                             hovertemplate='t = %{x:.1f} s<br>%{y:.2f} g<extra></extra>'), 1, 2)
    fig.add_trace(go.Scatter(x=[R['t'][kk]], y=[R['lat'][kk]], mode='markers', marker=dict(size=12, color=GD, line=dict(color='white', width=1.5)), visible=vis, showlegend=False,
                             hovertemplate=f'peak frame: {R["lat"][kk]:.2f} g<extra></extra>'), 1, 2)
    B20.append((r.display_name, f'{r.display_name}: highest Cut Force rep, {drill}, {r.g:.2f} g'))
btn = []
for k, (nm, title) in enumerate(B20):
    vis = [False] * (4 * len(B20)); vis[4 * k:4 * k + 4] = [True] * 4
    btn.append(dict(label=nm, method='update', args=[{'visible': vis}, {'title.text': title}]))
fig.update_layout(title=B20[0][1], updatemenus=[dict(buttons=btn, x=1.0, xanchor='right', y=1.25, yanchor='top', direction='down', bgcolor='white', font=dict(size=13))],
                  height=500, **{**LAY, 'margin': dict(l=60, r=20, t=120, b=50)})
fig.update_xaxes(title_text='Yards upfield', row=1, col=1); fig.update_yaxes(title_text='Yards across', scaleanchor='x', scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text='Seconds into the rep', row=1, col=2); fig.update_yaxes(title_text='Sideways acceleration (g)', rangemode='tozero', row=1, col=2)
show(fig)
print(f'{len(B20)} receivers in the menu, ordered by Combine Cut Force (highest first).')


# Section 5. Discussion and game implications

**What the results mean.** Cut Force is moderately reliable across nine controlled reps (0.59), independent of the 40 and the 3-cone, and visible again in NFL routes (transfer r = 0.23, n = 82), unlike braking or burst, which do not transfer. Receivers with higher Cut Force earn more targets per route after controls (odds ratio 1.13 per standard deviation, p = 0.062), although Nacua carries much of that result (without him, p = 0.20). The claim is therefore that Cut Force flags candidates, not sure things.

**For the draft room.** Cut Force flags receivers whose route ability may be underrated by timed drills. Nacua (pick 177) posted the highest Cut Force and a 33.6% target rate. Adonai Mitchell (pick 52) followed at 26.5%. Others above 1.5 standard deviations with NFL routes sit between 13.9% and 17.8%, around the 16.9% median (notebook section 10).

**For the weekly staff.** The formula needs only speed and direction of travel at 10 Hz, which game tracking provides and practice GPS units measure validly. Nine controlled reps reach a reliability of 0.59, while about 315 game routes reach only 0.43, so the weekly version belongs in practice route drills, not game film. In games, Cut Force held steady as running load built (section 12), so a practice drop is worth a closer look.

**For player development.** The drill-by-drill strip on the card (section 20) shows which breaks a receiver rounds. Even Nacua's slant and right-side post corner sat near the class middle, so a position coach knows which cuts to drill first.

**Where it stops.** Defensive line drill speed mostly repeats the 40 (section 11), and Cut Force does not transfer to defensive backs, whose turns follow the receiver (section 13). The metric is a receiver metric.

This ties back to the motivation: the Combine already records how hard a receiver can turn at speed, and this notebook turns those frames into one score a scout or a coach can use.


# Section 6. Limitations and future work

**Data.** The sample is 107 receivers, and 82 of them ran regular-season routes. The game tracking includes only rookie-cohort players, so most defenders and the ball are not available, so separation can be checked only on plays where a tracked defender is present (section 8). The 2023 to 2025 receivers were faster and more explosive than earlier classes (Figure 7, public nflverse data), so the class norms here may not hold for older classes.

**Model.** The study is observational. Teams may target receivers who show sharp cuts on film, and Cut Force may partly measure that same perception. The target result leans on the best example: with Nacua held out the effect stays positive but is no longer close to conventional significance. The measurement-error model rescales the slope, not the evidence for its sign (P(slope > 0) is 0.99 on both scales).

**Definition.** Cut Force uses a window of three frames either side of the turn peak at 10 Hz. Section 15 tests alternative definitions (first break, speed-weighted, peak variants); the first-break variant lowers held-out error the most, but only in the 2023 class. The choice of window is a judgment, and the sensitivity analysis is the honest report of it.

**Future work.**
1. More NFL seasons for the same receivers, to see how much Cut Force predicts targets once rookie-year noise washes out.
2. A practice-GPS version with a position group, to test whether the 0.59 reliability holds outside the Combine.
3. Separation as the outcome once defender tracking covers full rosters, since targets are a quarterback's decision and separation is the receiver's.
4. A tight end and running back version, because the route drills exist for both and the trench check suggests the metric is route specific.


# Section 7. References and reproducibility

**Reproducibility.** The notebook runs top to bottom on Kaggle with the competition data attached (about 17 to 20 minutes, mostly the PyMC fit in section 6 and the Plotly builds in sections 16 to 20). Every number in the writeup is printed by a cell here, and every writeup figure is written to `/kaggle/working/`. The only outside data is the public nflverse Combine release used for Figure 7 and Appendix G. Random seeds are set where sampling occurs. Version history: Version 16 (Nov 3, 2026) fixed Appendix H to the notebook's numbers, Version 20 (Nov 5) added the NFL season control, Version 21 (Nov 12) made every figure interactive, Version 22 (Nov 18) added the Big Data Bowl section map and the discussion, limitations and references cells, and Version 23 (Nov 18) reworded axis labels and legends in plain language. The notebook is public.

**References**

Abril-Pla, O., Andreani, V., Carroll, C., Dong, L., Fonnesbeck, C. J., Kochurov, M., Kumar, R., Lao, J., Luhmann, C. C., Martin, O. A., Osthege, M., Vieira, R., Wiecki, T., & Zinkov, R. (2023). PyMC: A modern, and comprehensive probabilistic programming framework in Python. *PeerJ Computer Science, 9*, e1516. https://doi.org/10.7717/peerj-cs.1516

Brown, W. (1910). Some experimental results in the correlation of mental abilities. *British Journal of Psychology, 3*(3), 296-322. https://doi.org/10.1111/j.2044-8295.1910.tb00207.x

Dos'Santos, T., Thomas, C., Comfort, P., & Jones, P. A. (2018). The effect of angle and velocity on change of direction biomechanics: An angle-velocity trade-off. *Sports Medicine, 48*(10), 2235-2253. https://doi.org/10.1007/s40279-018-0968-3

Efron, B., & Morris, C. (1975). Data analysis using Stein's estimator and its generalizations. *Journal of the American Statistical Association, 70*(350), 311-319. https://doi.org/10.1080/01621459.1975.10479864

Johnston, R. J., Watsford, M. L., Kelly, S. J., Pine, M. J., & Spurrs, R. W. (2014). Validity and interunit reliability of 10 Hz and 15 Hz GPS units for assessing athlete movement demands. *Journal of Strength and Conditioning Research, 28*(6), 1649-1655. https://doi.org/10.1519/JSC.0000000000000323

Lopez, M., Blake, A., Bliss, T., Howard, A., & Mooney, P. (2026). *NFL Big Data Bowl 2027* [Data set and competition]. Kaggle. https://www.kaggle.com/competitions/nfl-big-data-bowl-2027

Maaddi, R. (2026, January 10). Stafford gets first nod for AP All-Pro. Garrett, Nacua and Smith-Njigba are unanimous choices. *AP News*. https://apnews.com/article/nfl-allpro-team-86b72f1cb4283f16c9ae6b6908418d21

nflverse. (2026). *NFL Combine data courtesy of Pro Football Reference* (combine release) [Data set]. GitHub. https://github.com/nflverse/nflverse-data/releases/tag/combine

Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92-96. https://doi.org/10.25080/Majora-92bf1922-011

Spearman, C. (1904). The proof and measurement of association between two things. *American Journal of Psychology, 15*(1), 72-101. https://doi.org/10.2307/1412159

Spearman, C. (1910). Correlation calculated from faulty data. *British Journal of Psychology, 3*(3), 271-295. https://doi.org/10.1111/j.2044-8295.1910.tb00206.x

Ward, P. A., Ramsden, S., Coutts, A. J., Hulton, A. T., & Drust, B. (2018). Positional differences in running and nonrunning activities during elite American football training. *Journal of Strength and Conditioning Research, 32*(7), 2072-2084. https://doi.org/10.1519/JSC.0000000000002294
